# ⚽ Rastreamento Multi-Objeto de Jogadores em Vídeos de Futebol

**Trabalho de Conclusão de Curso — Engenharia Elétrica (Ifes, 2026)**

Detecção de jogadores com **YOLOv8** e comparação de **4 algoritmos de rastreamento multi-objeto** (ByteTrack, StrongSORT, OC-SORT e DeepSORT) sobre o dataset **SoccerNet 2022**, avaliando qual preserva melhor a identidade de cada jogador ao longo do vídeo.

---

## 1. Configuração do ambiente

### 1.1. Conexão com o Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

### 1.2. Download do dataset SoccerNet 2022 (tracking)

In [ ]:
from SoccerNet.Downloader import SoccerNetDownloader
mySoccerNetDownloader = SoccerNetDownloader(LocalDirectory="/content/drive/MyDrive/SoccerNet/tracking")
mySoccerNetDownloader.downloadDataTask(task="tracking", split=["train","test","challenge"])


### 1.3. Instalação do YOLOv8 (detecção)

In [ ]:
!pip install ultralytics

### 1.4. Instalação do BoxMOT (algoritmos de rastreamento: OC-SORT, StrongSORT, ByteTrack)

In [ ]:
!git clone https://github.com/mikel-brostrom/yolov8_tracking.git
%cd yolov8_tracking

In [ ]:
!pip install boxmot -q

### 1.5. Instalação do MotMetrics (cálculo da métrica MOTA)

In [ ]:
pip install motmetrics

In [ ]:
!pip install numpy==1.26.4 pandas==2.0.3 scipy==1.10.1 motmetrics==1.4.0 --upgrade --force-reinstall

## 2. Preparação dos dados

### 2.1. Conversão do SoccerNet para o formato YOLO

In [ ]:
import os
import shutil
import random
import configparser
import re

# Pastas de entrada e saída dos dados
DIR_TRACKING = '/content/drive/MyDrive/SoccerNet/tracking'
DIR_TRAIN = os.path.join(DIR_TRACKING, 'train')
DIR_TEST = os.path.join(DIR_TRACKING, 'test')
YOLO_SAIDA = '/content/yolo_dataset_cego'
YOLO_CLASSES = ["player", "goalkeeper", "referee", "ball"]

# Mapeando classes a partir do gameinfo.ini
def map_full_class_to_yolo_class(nome_classe):
    nome_classe_lower = nome_classe.lower()
    if "goalkeeper" in nome_classe_lower:
        return "goalkeeper"
    elif "player" in nome_classe_lower:
        return "player"
    elif "referee" in nome_classe_lower:
        return "referee"
    elif "ball" in nome_classe_lower:
        return "ball"
    return None

# Lendo gameinfo.ini e seqinfo.ini
def read_sequence_info(seq_folder_path):
    caminho_gameinfo = os.path.join(seq_folder_path, 'gameinfo.ini')
    caminho_seqinfo = os.path.join(seq_folder_path, 'seqinfo.ini')

    # Usando o config parser para ler os arquivos .ini
    config_parser = configparser.ConfigParser()

    # Lendo as dimensões do vídeo
    config_parser.read(caminho_seqinfo)
    img_width = int(config_parser['Sequence']['imWidth'])
    img_height = int(config_parser['Sequence']['imHeight'])

    # LER O GAME INFO
    config_parser.read(caminho_gameinfo)
    track_id_para_yolo_class= {}

    if 'Sequence' not in config_parser or 'num_tracklets' not in config_parser['Sequence']:
        print(f"Aviso: 'num_tracklets' não encontrado no gameinfo.ini de {seq_folder_path}.")
        return YOLO_CLASSES, img_width, img_height, track_id_para_yolo_class

    num_tracklets = int(config_parser['Sequence']['num_tracklets'])

    for i in range(1, num_tracklets + 1):
        tracklet_key = f'trackletID_{i}'
        if tracklet_key not in config_parser['Sequence']:
            print(f"Aviso: {tracklet_key} não encontrado no gameinfo.ini de {seq_folder_path}. Pulando tracklet {i}.")
            continue

        tracklet_entry = config_parser['Sequence'][tracklet_key]
        parts = tracklet_entry.split(';', 1)
        if len(parts) != 2:
            print(f"Aviso: Entrada '{tracklet_entry}' mal formatada em {seq_folder_path}. Pulando tracklet {i}.")
            continue

        full_class_name = parts[0].strip()
        simple_yolo_class = map_full_class_to_yolo_class(full_class_name)

        if simple_yolo_class:
            track_id_para_yolo_class[str(i)] = simple_yolo_class

    return YOLO_CLASSES, img_width, img_height, track_id_para_yolo_class

# Função para converter coordenadas para o formato YOLO
def conversao_bbox_yolo(xtl, ytl, xbr, ybr, img_width, img_height):
    dw = 1.0 / img_width
    dh = 1.0 / img_height
    x_center = (xtl + xbr) / 2.0 * dw
    y_center = (ytl + ybr) / 2.0 * dh
    width_bbox = (xbr - xtl) * dw
    height_bbox = (ybr - ytl) * dh
    x_center = max(0.0, min(1.0, x_center))
    y_center = max(0.0, min(1.0, y_center))
    width_bbox = max(0.000001, min(1.0, width_bbox))
    height_bbox = max(0.000001, min(1.0, height_bbox))
    return x_center, y_center, width_bbox, height_bbox

# Lendo o Ground Truth
def read_gt_file(file_path):
    deteccoes = []
    with open(file_path, 'r') as f:
        for line_num, line in enumerate(f.readlines()):
            values = line.strip().split(',')
            try:
                frame_id = int(values[0])
                track_id_str = values[1].strip()
                xtl = float(values[2])
                ytl = float(values[3])
                width = float(values[4])
                height = float(values[5])

                if width <= 0 or height <= 0:
                    continue

                deteccoes.append((frame_id, track_id_str, xtl, ytl, width, height))
            except ValueError:
                print(f"Aviso: ValueError ao processar linha {line_num+1} em {file_path}.")
                continue
    return deteccoes

# Processa GT para labels YOLO
def process_gt_to_yolo_labels(gt_deteccoes_for_sequence, yolo_target_classes_ordered, tracking_and_txt_output, img_width, img_height, yolo_image_filename, yolo_label_dir):
    match = re.search(r'_(\d{1,6})\.jpg$', yolo_image_filename)
    if not match:
        return

    current_frame_id_of_image = int(match.group(1)) # <-- CORREÇÃO AQUI para não travar
    label_file_path = os.path.join(yolo_label_dir, os.path.splitext(yolo_image_filename)[0] + '.txt')
    yolo_labels_for_this_image = []

    for frame_id_from_gt, track_id_str_from_gt, xtl, ytl, width, height in gt_deteccoes_for_sequence:
        if frame_id_from_gt == current_frame_id_of_image:
            simple_yolo_class = tracking_and_txt_output.get(track_id_str_from_gt)
            try:
                yolo_class_index = yolo_target_classes_ordered.index(simple_yolo_class)
            except ValueError:
                continue

            xbr = xtl + width
            ybr = ytl + height
            x_center, y_center, w_yolo, h_yolo = conversao_bbox_yolo(xtl, ytl, xbr, ybr, img_width, img_height)
            yolo_labels_for_this_image.append(f"{yolo_class_index} {x_center:.6f} {y_center:.6f} {w_yolo:.6f} {h_yolo:.6f}\n")

    if yolo_labels_for_this_image:
        with open(label_file_path, 'w') as f:
            f.writelines(yolo_labels_for_this_image)

# Função modificada para aceitar uma lista específica de pastas (Evitar Data Leakage)
def process_sequences_list(source_sequences_parent_dir, sequence_names_list, set_name, yolo_img_output_dir, yolo_lbl_output_dir):
    print(f"\nProcessando {len(sequence_names_list)} Sequências para o Conjunto: {set_name}")
    processed_sequences_count_for_set = 0

    for sn_folder_name in sequence_names_list:
        sn_folder_path = os.path.join(source_sequences_parent_dir, sn_folder_name)
        try:
            yolo_classes_ordered, img_w, img_h, trackid_map_to_yolo_class = read_sequence_info(sn_folder_path)
        except Exception as e:
            print(f"Erro ao ler a sequência {sn_folder_name}. Pulando sequência.")
            continue

        gt_file_path = os.path.join(sn_folder_path, 'gt', 'gt.txt')
        all_gt_deteccoes_for_sequence = read_gt_file(gt_file_path)

        img1_folder_path = os.path.join(sn_folder_path, 'img1')
        original_image_filenames = sorted([f for f in os.listdir(img1_folder_path) if f.endswith('.jpg')])

        for orig_img_name in original_image_filenames:
            yolo_img_filename = f"{sn_folder_name}_{orig_img_name}"
            try:
                shutil.copy(os.path.join(img1_folder_path, orig_img_name), os.path.join(yolo_img_output_dir, yolo_img_filename))
                if all_gt_deteccoes_for_sequence:
                    process_gt_to_yolo_labels(all_gt_deteccoes_for_sequence, yolo_classes_ordered,
                                              trackid_map_to_yolo_class, img_w, img_h,
                                              yolo_img_filename, yolo_lbl_output_dir)
            except Exception as e:
                print(f" Erro ao processar/copiar imagem {orig_img_name}: {e}")
        processed_sequences_count_for_set += 1
    return processed_sequences_count_for_set


# Função Principal Refeita para o Split Correto
def generate_yolo_dataset_from_official_splits():
    print(f"Iniciando geração do dataset YOLO com particionamento cego (sem viés).")

    # Criando estrutura YOLO (train, val, test)
    train_img_dir = os.path.join(YOLO_SAIDA, 'train', 'images')
    train_lbl_dir = os.path.join(YOLO_SAIDA, 'train', 'labels')
    val_img_dir = os.path.join(YOLO_SAIDA, 'val', 'images')
    val_lbl_dir = os.path.join(YOLO_SAIDA, 'val', 'labels')
    test_img_dir = os.path.join(YOLO_SAIDA, 'test', 'images')
    test_lbl_dir = os.path.join(YOLO_SAIDA, 'test', 'labels')

    dirs_to_make = [train_img_dir, train_lbl_dir, val_img_dir, val_lbl_dir, test_img_dir, test_lbl_dir]
    for dir_path in dirs_to_make:
        if os.path.exists(dir_path):
            shutil.rmtree(dir_path)
        os.makedirs(dir_path, exist_ok=True)

    # 1. PEGAR AS 57 SEQUÊNCIAS DE TREINO E FATIAR (45 Treino / 12 Validação)
    todas_seq_treino = [name for name in os.listdir(DIR_TRAIN) if os.path.isdir(os.path.join(DIR_TRAIN, name)) and name.startswith("SNMOT-")]

    # Semente fixa para sempre separar os mesmos vídeos se você precisar rodar de novo
    random.seed(42)
    random.shuffle(todas_seq_treino)

    # Fazendo o split (Aproximadamente 80% treino, 20% validação)
    qtd_treino = int(len(todas_seq_treino) * 0.8)
    lista_treino = todas_seq_treino[:qtd_treino]
    lista_validacao = todas_seq_treino[qtd_treino:]

    # 2. PEGAR AS 49 SEQUÊNCIAS DE TESTE (Ficam intocadas para o final)
    lista_teste = [name for name in os.listdir(DIR_TEST) if os.path.isdir(os.path.join(DIR_TEST, name)) and name.startswith("SNMOT-")]

    # 3. EXECUTAR O PROCESSAMENTO
    train_processed = process_sequences_list(DIR_TRAIN, lista_treino, "Treino (YOLO train)", train_img_dir, train_lbl_dir)
    val_processed = process_sequences_list(DIR_TRAIN, lista_validacao, "Validação (YOLO val)", val_img_dir, val_lbl_dir)
    test_processed = process_sequences_list(DIR_TEST, lista_teste, "Teste Cego (YOLO test)", test_img_dir, test_lbl_dir)

    print(f"\nProcessamento concluído com sucesso.")
    print(f"  {train_processed} sequências separadas para TREINO.")
    print(f"  {val_processed} sequências separadas para VALIDAÇÃO.")
    print(f"  {test_processed} sequências intocadas para TESTE FINAL.")

if __name__ == '__main__':
    generate_yolo_dataset_from_official_splits()

In [ ]:
!zip -r -q /content/drive/MyDrive/SoccerNet/yolo_dataset_cego_final.zip /content/yolo_dataset_cego

### 2.2. Descompactação do dataset no ambiente local do Colab

### 2.3. Descompactação e preparação para o treino

In [ ]:
import os
import yaml
from ultralytics import YOLO

# 1. DESCOMPACTAÇÃO CORRIGIDA (Forçando sobrescrever e na raiz correta)
zip_no_drive = "/content/drive/MyDrive/SoccerNet/yolo_dataset_cego_final.zip"

print("Limpando pastas e descompactando dataset certinho...")
# O '-o' significa 'overwrite' (sobrescrever sem perguntar)
# O '-d /' garante que não vai duplicar a pasta 'content'
!unzip -o -q {zip_no_drive} -d /
print("Dataset descompactado com sucesso na pasta correta!")

# 2. CRIAÇÃO DO YAML BLINDADO
yaml_file_path = '/content/soccernet_yolo_cego.yaml'
yaml_content = {
    'path': '/content/yolo_dataset_cego',
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    'nc': 4,
    'names': ["player", "goalkeeper", "referee", "ball"]
}

with open(yaml_file_path, 'w') as f:
    yaml.dump(yaml_content, f, sort_keys=False, default_flow_style=False)
print(f"YAML criado em: {yaml_file_path}")

# 3. TREINAMENTO
model = YOLO('yolov8s.pt')

results = model.train(
    data=yaml_file_path,
    epochs=50,
    imgsz=640,
    batch=16,
    project='/content/drive/MyDrive/SoccerNet/runs',
    name='treino_cego_oficial'
)
print("Treinamento concluído e salvo com sucesso no Drive!")

## 3. Treinamento do detector

### 3.1. Data augmentation (Albumentations) e treino do YOLOv8

In [ ]:
import os
import shutil
import configparser
import re
import cv2
import albumentations as A
import numpy as np
import random
from google.colab import drive

# Monta o Drive se já não estiver montado
drive.mount('/content/drive', force_remount=True)

# Pastas originais no seu Drive
SOCCERNET_TRACKING_ROOT_DIR = '/content/drive/MyDrive/SoccerNet/tracking'
SOURCE_YOLO_TRAIN_SEQUENCES_DIR = os.path.join(SOCCERNET_TRACKING_ROOT_DIR, 'train')
SOURCE_YOLO_TEST_SEQUENCES_DIR = os.path.join(SOCCERNET_TRACKING_ROOT_DIR, 'test')

# SAÍDA LOCAL: Salvar no SSD do Colab para ser 100x mais rápido e não dar erro de I/O
YOLO_DATASET_AUGMENTED_DIR = '/content/yolo_augmented_cego'
YOLO_TARGET_CLASSES = ["player", "goalkeeper", "referee", "ball"]

# Ativação da augmentation apenas para o treino
AUGMENT_TRAIN_SET = True

def map_full_class_to_yolo_class(full_class_name_str):
    fcn_lower = full_class_name_str.lower()
    if "goalkeeper" in fcn_lower: return "goalkeeper"
    elif "player" in fcn_lower: return "player"
    elif "referee" in fcn_lower: return "referee"
    elif "ball" in fcn_lower: return "ball"
    return None

def convert_to_yolo_bbox(bbox, img_width, img_height):
    xtl, ytl, xbr, ybr = bbox
    dw = 1.0 / img_width
    dh = 1.0 / img_height
    x_center = (xtl + xbr) / 2.0 * dw
    y_center = (ytl + ybr) / 2.0 * dh
    width_bbox = (xbr - xtl) * dw
    height_bbox = (ybr - ytl) * dh
    return max(0.0, min(1.0, x_center)), max(0.0, min(1.0, y_center)), max(0.001, min(1.0, width_bbox)), max(0.001, min(1.0, height_bbox))

def get_augmentation_pipeline():
    return A.Compose([
        A.CoarseDropout(max_holes=8, max_height=80, max_width=80, p=0.5),
        A.RandomBrightnessContrast(p=0.7),
        A.MotionBlur(p=0.5),
        A.GaussNoise(p=0.4),
        A.HorizontalFlip(p=0.5),
    ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['class_labels'], min_visibility=0.2))

def read_sequence_info(seq_folder_path):
    caminho_gameinfo = os.path.join(seq_folder_path, 'gameinfo.ini')
    caminho_seqinfo = os.path.join(seq_folder_path, 'seqinfo.ini')
    config_parser = configparser.ConfigParser()
    config_parser.read(caminho_seqinfo)
    img_width = int(config_parser['Sequence']['imWidth'])
    img_height = int(config_parser['Sequence']['imHeight'])

    config_parser.read(caminho_gameinfo)
    track_id_map = {}
    if 'Sequence' not in config_parser or 'num_tracklets' not in config_parser['Sequence']:
        return YOLO_TARGET_CLASSES, img_width, img_height, track_id_map

    num_tracklets = int(config_parser['Sequence']['num_tracklets'])
    for i in range(1, num_tracklets + 1):
        tracklet_key = f'trackletID_{i}'
        if tracklet_key not in config_parser['Sequence']: continue

        tracklet_entry = config_parser['Sequence'][tracklet_key]
        parts = tracklet_entry.split(';', 1)
        if len(parts) != 2: continue

        full_class_name = parts[0].strip()
        simple_yolo_class = map_full_class_to_yolo_class(full_class_name)

        if simple_yolo_class:
            track_id_map[str(i)] = simple_yolo_class

    return YOLO_TARGET_CLASSES, img_width, img_height, track_id_map

# Função modificada para aceitar a nossa lista fatiada de sequências (Data Leakage blindado)
def process_sequences_list(source_dir, sequence_names_list, set_name, img_output_dir, lbl_output_dir, augment=False):
    print(f"\nProcessando {len(sequence_names_list)} Sequências para o Conjunto: {set_name}")
    if augment:
        print(">>>>> AUGMENTATION ATIVADA <<<<<")
        aug_pipeline = get_augmentation_pipeline()

    for sn_folder_name in sequence_names_list:
        sn_folder_path = os.path.join(source_dir, sn_folder_name)
        print(f"  Processando {sn_folder_name}...")
        try:
            _, img_w, img_h, trackid_map = read_sequence_info(sn_folder_path)
        except Exception as e:
            continue

        gt_file_path = os.path.join(sn_folder_path, 'gt', 'gt.txt')
        if not os.path.exists(gt_file_path): continue

        detections_by_frame = {}
        with open(gt_file_path, 'r') as f:
            for line in f.readlines():
                values = line.strip().split(',')
                if len(values) < 6: continue
                frame_id, track_id, xtl, ytl, w, h = int(values[0]), values[1].strip(), float(values[2]), float(values[3]), float(values[4]), float(values[5])
                if frame_id not in detections_by_frame:
                    detections_by_frame[frame_id] = []
                detections_by_frame[frame_id].append({'track_id': track_id, 'bbox': [xtl, ytl, xtl + w, ytl + h]})

        img1_folder_path = os.path.join(sn_folder_path, 'img1')
        if not os.path.isdir(img1_folder_path): continue

        for orig_img_name in sorted(os.listdir(img1_folder_path)):
            if not orig_img_name.endswith('.jpg'): continue
            match = re.search(r'(\d{6})\.jpg$', orig_img_name)
            if not match: continue
            frame_id = int(match.group(1))

            bboxes_for_frame = []
            class_labels_for_frame = []
            if frame_id in detections_by_frame:
                for det in detections_by_frame[frame_id]:
                    yolo_class = trackid_map.get(det['track_id'])
                    if yolo_class:
                        bboxes_for_frame.append(det['bbox'])
                        class_labels_for_frame.append(yolo_class)

            image_path = os.path.join(img1_folder_path, orig_img_name)
            image = cv2.imread(image_path)
            if image is None: continue
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

            yolo_labels = []

            if augment and bboxes_for_frame:
                try:
                    transformed = aug_pipeline(image=image, bboxes=bboxes_for_frame, class_labels=class_labels_for_frame)
                    image_to_save = transformed['image']
                    for bbox, label in zip(transformed['bboxes'], transformed['class_labels']):
                        yolo_bbox = convert_to_yolo_bbox(bbox, img_w, img_h)
                        class_index = YOLO_TARGET_CLASSES.index(label)
                        yolo_labels.append(f"{class_index} {yolo_bbox[0]:.6f} {yolo_bbox[1]:.6f} {yolo_bbox[2]:.6f} {yolo_bbox[3]:.6f}")
                except Exception as e:
                    image_to_save = image
                    for bbox, label in zip(bboxes_for_frame, class_labels_for_frame):
                        yolo_bbox = convert_to_yolo_bbox(bbox, img_w, img_h)
                        class_index = YOLO_TARGET_CLASSES.index(label)
                        yolo_labels.append(f"{class_index} {yolo_bbox[0]:.6f} {yolo_bbox[1]:.6f} {yolo_bbox[2]:.6f} {yolo_bbox[3]:.6f}")
            else:
                image_to_save = image
                for bbox, label in zip(bboxes_for_frame, class_labels_for_frame):
                    yolo_bbox = convert_to_yolo_bbox(bbox, img_w, img_h)
                    class_index = YOLO_TARGET_CLASSES.index(label)
                    yolo_labels.append(f"{class_index} {yolo_bbox[0]:.6f} {yolo_bbox[1]:.6f} {yolo_bbox[2]:.6f} {yolo_bbox[3]:.6f}")

            yolo_img_filename = f"{sn_folder_name}_{orig_img_name}"
            output_img_path = os.path.join(img_output_dir, yolo_img_filename)
            output_lbl_path = os.path.join(lbl_output_dir, os.path.splitext(yolo_img_filename)[0] + '.txt')

            image_to_save_bgr = cv2.cvtColor(image_to_save, cv2.COLOR_RGB2BGR)
            cv2.imwrite(output_img_path, image_to_save_bgr)

            if yolo_labels:
                with open(output_lbl_path, 'w') as f:
                    f.write("\n".join(yolo_labels))

def generate_yolo_dataset_robust_cego():
    print(f"Iniciando geração do dataset Augmented com Particionamento Cego em: {YOLO_DATASET_AUGMENTED_DIR}")

    train_img_dir = os.path.join(YOLO_DATASET_AUGMENTED_DIR, 'train', 'images')
    train_lbl_dir = os.path.join(YOLO_DATASET_AUGMENTED_DIR, 'train', 'labels')
    val_img_dir = os.path.join(YOLO_DATASET_AUGMENTED_DIR, 'val', 'images')
    val_lbl_dir = os.path.join(YOLO_DATASET_AUGMENTED_DIR, 'val', 'labels')
    test_img_dir = os.path.join(YOLO_DATASET_AUGMENTED_DIR, 'test', 'images')
    test_lbl_dir = os.path.join(YOLO_DATASET_AUGMENTED_DIR, 'test', 'labels')

    if os.path.exists(YOLO_DATASET_AUGMENTED_DIR):
        shutil.rmtree(YOLO_DATASET_AUGMENTED_DIR)
    for d in [train_img_dir, train_lbl_dir, val_img_dir, val_lbl_dir, test_img_dir, test_lbl_dir]:
        os.makedirs(d, exist_ok=True)

    # 1. PEGAR AS 57 SEQUÊNCIAS DE TREINO E FATIAR (45 Treino / 12 Validação)
    todas_seq_treino = [name for name in os.listdir(SOURCE_YOLO_TRAIN_SEQUENCES_DIR) if os.path.isdir(os.path.join(SOURCE_YOLO_TRAIN_SEQUENCES_DIR, name)) and name.startswith("SNMOT-")]

    random.seed(42)
    random.shuffle(todas_seq_treino)

    qtd_treino = int(len(todas_seq_treino) * 0.8)
    lista_treino = todas_seq_treino[:qtd_treino]
    lista_validacao = todas_seq_treino[qtd_treino:]

    # 2. PEGAR AS 49 SEQUÊNCIAS DE TESTE (Ficam intocadas)
    lista_teste = [name for name in os.listdir(SOURCE_YOLO_TEST_SEQUENCES_DIR) if os.path.isdir(os.path.join(SOURCE_YOLO_TEST_SEQUENCES_DIR, name)) and name.startswith("SNMOT-")]

    # 3. EXECUTAR O PROCESSAMENTO (Augmentation apenas no treino)
    process_sequences_list(SOURCE_YOLO_TRAIN_SEQUENCES_DIR, lista_treino, "Treino (YOLO train)", train_img_dir, train_lbl_dir, augment=AUGMENT_TRAIN_SET)
    process_sequences_list(SOURCE_YOLO_TRAIN_SEQUENCES_DIR, lista_validacao, "Validação (YOLO val)", val_img_dir, val_lbl_dir, augment=False)
    process_sequences_list(SOURCE_YOLO_TEST_SEQUENCES_DIR, lista_teste, "Teste Cego (YOLO test)", test_img_dir, test_lbl_dir, augment=False)

    print("\nProcessamento concluído com sucesso!")

if __name__ == '__main__':
    generate_yolo_dataset_robust_cego()

In [ ]:
import os
import yaml
from google.colab import drive
from ultralytics import YOLO

# 1. MONTAR O DRIVE
drive.mount('/content/drive', force_remount=True)

# 2. DESCOMPACTAR O DATASET (Direto na raiz para manter a estrutura /content/yolo_augmented_cego)
zip_path = "/content/drive/MyDrive/SoccerNet/dataset_final_pronto.zip"
print("Descompactando dataset para o SSD da GPU. Isso pode levar alguns minutos...")
!unzip -o -q {zip_path} -d /
print("Dataset pronto no SSD local!")

# 3. CRIAR O YAML DE TREINAMENTO
yaml_file_path = '/content/soccernet_yolo_cego.yaml'
yaml_content = {
    'path': '/content/yolo_augmented_cego',
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    'nc': 4,
    'names': ["player", "goalkeeper", "referee", "ball"]
}

with open(yaml_file_path, 'w') as f:
    yaml.dump(yaml_content, f, sort_keys=False, default_flow_style=False)
print(f"YAML criado em: {yaml_file_path}")

# 4. INICIAR O TREINAMENTO (Com YOLO Augmentation DESLIGADA)
print("Iniciando o treinamento do YOLOv8 na A100...")
model = YOLO('yolov8s.pt')

results = model.train(
    data=yaml_file_path,
    epochs=50,
    imgsz=640,
    batch=32, # A100 aguenta batch 32 rindo. Vai acelerar muito!
    project='/content/drive/MyDrive/SoccerNet/runs',
    name='treino_cego_albumentations',

    # --- DESLIGANDO A AUGMENTATION NATIVA DO YOLO ---
    # Como o seu Albumentations já fez isso, zeramos tudo aqui para evitar "dupla distorção"
    hsv_h=0.0, hsv_s=0.0, hsv_v=0.0,
    degrees=0.0, translate=0.0, scale=0.0,
    shear=0.0, perspective=0.0,
    flipud=0.0, fliplr=0.0,
    mosaic=0.0, mixup=0.0, copy_paste=0.0
)
print("Treinamento concluído com sucesso!")

## 4. Rastreamento — Fase 1 (otimização do limiar de confiança)

### 4.1. OC-SORT

In [ ]:
import os
import cv2
import torch
import numpy as np
from ultralytics import YOLO
from boxmot import OcSort

# 1. DIRETÓRIOS
# Apontamos para o 'train' original, MAS vamos filtrar apenas os 12 do Valid logo abaixo!
BASE_INPUT_DIR = '/content/drive/MyDrive/SoccerNet/tracking/train/'
YOLO_MODEL_PATH = '/content/drive/MyDrive/SoccerNet/runs/treino_cego_albumentations/weights/best.pt'
BASE_OUTPUT_DIR = '/content/drive/MyDrive/SoccerNet/tracking_fase1_confianca/'

# 2. A SUA LISTA DE VALIDAÇÃO (O Segredo para não viciar o modelo)
sequence_folders = [
    "SNMOT-113", "SNMOT-157", "SNMOT-066", "SNMOT-167",
    "SNMOT-068", "SNMOT-074", "SNMOT-075", "SNMOT-077",
    "SNMOT-161", "SNMOT-061", "SNMOT-067", "SNMOT-154"
]

# Os três limiares do YOLO que vamos testar (da sua Tabela 2)
lista_confidences = [0.05, 0.3, 0.5]

# Parâmetros PADRÃO do OcSORT (Congelados para esta fase)
OCSORT_MAX_AGE = 30
OCSORT_MIN_HITS = 3
OCSORT_ASSO = 0.3

def run_fase1_ocsort():
    DEVICE = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')
    print("Carregando o modelo YOLO best.pt...")
    yolo_model = YOLO(YOLO_MODEL_PATH)
    yolo_model.to(DEVICE)

    # LOOP 1: Testa cada limiar de confiança do YOLO
    for conf_thresh in lista_confidences:

        tracker_name = f'OcSORT_Padrao_thr{conf_thresh}'
        output_txt_dir = os.path.join(BASE_OUTPUT_DIR, tracker_name, 'txt_files')
        os.makedirs(output_txt_dir, exist_ok=True)

        print(f"\n=======================================================")
        print(f"FASE 1 - TESTANDO CONFIANÇA YOLO: {conf_thresh}")
        print(f"=======================================================\n")

        # LOOP 2: Processa APENAS os 12 vídeos de validação
        for seq_folder_name in sequence_folders:
            print(f"--- Processando a sequência: {seq_folder_name} ---")

            # Instancia o tracker (o 'det_thresh' muda a cada rodada do loop)
            tracker_instance = OcSort(
                det_thresh=conf_thresh,
                max_age=OCSORT_MAX_AGE,
                min_hits=OCSORT_MIN_HITS,
                asso_threshold=OCSORT_ASSO
            )

            current_image_dir = os.path.join(BASE_INPUT_DIR, seq_folder_name, 'img1')
            output_txt_path = os.path.join(output_txt_dir, f"{seq_folder_name}.txt")

            # Verifica se a pasta existe antes de tentar ler
            if not os.path.exists(current_image_dir):
                print(f"ERRO: Pasta {current_image_dir} não encontrada. Pulando...")
                continue

            image_files = sorted([f for f in os.listdir(current_image_dir) if f.lower().endswith('.jpg')])
            if not image_files: continue

            all_tracks_for_seq = []

            for frame_num, image_name in enumerate(image_files):
                # Print de progresso para você saber que não travou
                if (frame_num + 1) % 200 == 0:
                    print(f"   ...frame {frame_num + 1}/{len(image_files)}")

                frame = cv2.imread(os.path.join(current_image_dir, image_name))
                if frame is None: continue

                # O YOLO detecta usando o limiar atual
                yolo_results = yolo_model.predict(frame, device=DEVICE, verbose=False, conf=conf_thresh)
                detections = yolo_results[0].boxes.data.cpu().numpy()

                if detections.shape[0] > 0:
                    tracks = tracker_instance.update(detections, frame)
                else:
                    tracks = tracker_instance.update(np.empty((0, 6)), frame)

                if len(tracks) > 0:
                    for track in tracks:
                        x1, y1, x2, y2, track_id, conf, cls, idx = track
                        all_tracks_for_seq.append(
                            f"{frame_num + 1},{int(track_id)},{x1:.2f},{y1:.2f},{x2-x1:.2f},{y2-y1:.2f},{conf:.6f},-1,-1,-1\n"
                        )

            with open(output_txt_path, 'w') as f:
                f.writelines(all_tracks_for_seq)

        print(f"-> Teste com Confiança {conf_thresh} concluído e salvo no Drive!")

if __name__ == '__main__':
    run_fase1_ocsort()
    print("\nTODOS OS TESTES DA FASE 1 TERMINADOS!")

### 4.2. StrongSORT

In [ ]:
import os
import cv2
import torch
import numpy as np
from pathlib import Path
from ultralytics import YOLO
from boxmot import StrongSort

# 1. DIRETÓRIOS
# Apontamos para o 'train' original, MAS vamos filtrar apenas os 12 do Valid logo abaixo!
BASE_INPUT_DIR = '/content/drive/MyDrive/SoccerNet/tracking/train/'
YOLO_MODEL_PATH = '/content/drive/MyDrive/SoccerNet/runs/treino_cego_albumentations/weights/best.pt'
BASE_OUTPUT_DIR = '/content/drive/MyDrive/SoccerNet/tracking_fase1_strongsort/' # Pasta nova para não misturar com OcSORT
REID_MODEL_PATH = 'osnet_x0_25_msmt17.pt' # Modelo extra que o StrongSORT exige

# 2. A SUA LISTA DE VALIDAÇÃO (O Segredo para não viciar o modelo)
sequence_folders = [
    "SNMOT-113", "SNMOT-157", "SNMOT-066", "SNMOT-167",
    "SNMOT-068", "SNMOT-074", "SNMOT-075", "SNMOT-077",
    "SNMOT-161", "SNMOT-061", "SNMOT-067", "SNMOT-154"
]

# Os três limiares do YOLO que vamos testar (da sua Tabela 2)
lista_confidences = [0.05, 0.3, 0.5]

# Parâmetros PADRÃO do StrongSORT (Congelados para esta fase, baseados na sua Tabela 1)
STRONGSORT_MAX_AGE = 70
STRONGSORT_N_INIT = 3
STRONGSORT_MAX_COS_DIST = 0.2

def run_fase1_strongsort():
    DEVICE = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')
    print("Carregando o modelo YOLO best.pt...")
    yolo_model = YOLO(YOLO_MODEL_PATH)
    yolo_model.to(DEVICE)

    # LOOP 1: Testa cada limiar de confiança do YOLO
    for conf_thresh in lista_confidences:

        tracker_name = f'StrongSORT_Padrao_thr{conf_thresh}'
        output_txt_dir = os.path.join(BASE_OUTPUT_DIR, tracker_name, 'txt_files')
        os.makedirs(output_txt_dir, exist_ok=True)

        print(f"\n=======================================================")
        print(f"FASE 1 - TESTANDO CONFIANÇA YOLO COM STRONGSORT: {conf_thresh}")
        print(f"=======================================================\n")

        # LOOP 2: Processa APENAS os 12 vídeos de validação
        for seq_folder_name in sequence_folders:
            print(f"--- Processando a sequência: {seq_folder_name} ---")

            # Instancia o tracker StrongSORT
            tracker_instance = StrongSort(
                reid_weights=Path(REID_MODEL_PATH),
                device=DEVICE,
                half=False,
                max_age=STRONGSORT_MAX_AGE,
                n_init=STRONGSORT_N_INIT,
                max_cos_dist=STRONGSORT_MAX_COS_DIST
            )

            current_image_dir = os.path.join(BASE_INPUT_DIR, seq_folder_name, 'img1')
            output_txt_path = os.path.join(output_txt_dir, f"{seq_folder_name}.txt")

            # Verifica se a pasta existe antes de tentar ler
            if not os.path.exists(current_image_dir):
                print(f"ERRO: Pasta {current_image_dir} não encontrada. Pulando...")
                continue

            image_files = sorted([f for f in os.listdir(current_image_dir) if f.lower().endswith('.jpg')])
            if not image_files: continue

            all_tracks_for_seq = []

            for frame_num, image_name in enumerate(image_files):
                # Print de progresso para você saber que não travou
                if (frame_num + 1) % 200 == 0:
                    print(f"   ...frame {frame_num + 1}/{len(image_files)}")

                frame = cv2.imread(os.path.join(current_image_dir, image_name))
                if frame is None: continue

                # O YOLO detecta usando o limiar atual
                yolo_results = yolo_model.predict(frame, device=DEVICE, verbose=False, conf=conf_thresh)
                detections = yolo_results[0].boxes.data.cpu().numpy()

                # StrongSORT precisa de uma atenção especial se não houver detecções
                if detections.shape[0] > 0:
                    tracks = tracker_instance.update(detections, frame)
                else:
                    tracks = tracker_instance.update(np.empty((0, 6)), frame)

                if len(tracks) > 0:
                    for track in tracks:
                        x1, y1, x2, y2, track_id, conf, cls, idx = track
                        all_tracks_for_seq.append(
                            f"{frame_num + 1},{int(track_id)},{x1:.2f},{y1:.2f},{x2-x1:.2f},{y2-y1:.2f},{conf:.6f},-1,-1,-1\n"
                        )

            with open(output_txt_path, 'w') as f:
                f.writelines(all_tracks_for_seq)

        print(f"-> Teste StrongSORT com Confiança {conf_thresh} concluído e salvo no Drive!")

if __name__ == '__main__':
    run_fase1_strongsort()
    print("\nTODOS OS TESTES DA FASE 1 DO STRONGSORT TERMINADOS!")

### 4.3. ByteTrack

In [ ]:
import os
import cv2
import torch
import numpy as np
from ultralytics import YOLO
from boxmot import ByteTrack

# 1. DIRETÓRIOS DA VALIDAÇÃO (O mesmo esquema blindado)
BASE_INPUT_DIR = '/content/drive/MyDrive/SoccerNet/tracking/train/'
YOLO_MODEL_PATH = '/content/drive/MyDrive/SoccerNet/runs/treino_cego_albumentations/weights/best.pt'
BASE_OUTPUT_DIR = '/content/drive/MyDrive/SoccerNet/tracking_fase1_bytetrack/' # Pasta exclusiva

# 2. A SUA LISTA DE VALIDAÇÃO DOS 12 VÍDEOS
sequence_folders = [
    "SNMOT-113", "SNMOT-157", "SNMOT-066", "SNMOT-167",
    "SNMOT-068", "SNMOT-074", "SNMOT-075", "SNMOT-077",
    "SNMOT-161", "SNMOT-061", "SNMOT-067", "SNMOT-154"
]

# Os três limiares do YOLO que vamos testar
lista_confidences = [0.05, 0.3, 0.5]

# Parâmetros PADRÃO do ByteTrack (Baseados na sua Tabela 1)
BYTETRACK_TRACK_BUFFER = 30  # Equivale ao max_age
BYTETRACK_TRACK_THRESH = 0.6 # Equivale ao n_init / track_thresh
BYTETRACK_MATCH_THRESH = 0.8 # Equivale ao asso / match / cos_dist

def run_fase1_bytetrack():
    DEVICE = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')
    print("Carregando o modelo YOLO best.pt...")
    yolo_model = YOLO(YOLO_MODEL_PATH)
    yolo_model.to(DEVICE)

    # LOOP 1: Testa cada limiar de confiança do YOLO
    for conf_thresh in lista_confidences:

        tracker_name = f'ByteTrack_Padrao_thr{conf_thresh}'
        output_txt_dir = os.path.join(BASE_OUTPUT_DIR, tracker_name, 'txt_files')
        os.makedirs(output_txt_dir, exist_ok=True)

        print(f"\n=======================================================")
        print(f"FASE 1 - TESTANDO CONFIANÇA YOLO COM BYTETRACK: {conf_thresh}")
        print(f"=======================================================\n")

        # LOOP 2: Processa APENAS os 12 vídeos de validação
        for seq_folder_name in sequence_folders:
            print(f"--- Processando Sequência: {seq_folder_name} ---")

            # Inicializa a instância do ByteTrack (Ele não precisa de ReID!)
            tracker_instance = ByteTrack(
                track_thresh=BYTETRACK_TRACK_THRESH,
                track_buffer=BYTETRACK_TRACK_BUFFER,
                match_thresh=BYTETRACK_MATCH_THRESH
            )

            current_image_dir = os.path.join(BASE_INPUT_DIR, seq_folder_name, 'img1')
            output_txt_path = os.path.join(output_txt_dir, f"{seq_folder_name}.txt")

            if not os.path.exists(current_image_dir):
                print(f"ERRO: Pasta {current_image_dir} não encontrada. Pulando...")
                continue

            image_files = sorted([f for f in os.listdir(current_image_dir) if f.lower().endswith('.jpg')])
            if not image_files: continue

            all_tracks_for_seq = []

            for frame_num, image_name in enumerate(image_files):
                # Imprime progresso
                if (frame_num + 1) % 200 == 0:
                    print(f"   ...processando frame {frame_num + 1}/{len(image_files)}")

                frame = cv2.imread(os.path.join(current_image_dir, image_name))
                if frame is None: continue

                # YOLO detecta usando o limiar atual da rodada
                yolo_results = yolo_model.predict(frame, device=DEVICE, verbose=False, conf=conf_thresh)
                detections = yolo_results[0].boxes.data.cpu().numpy()

                # Alimenta o tracker
                if detections.shape[0] > 0:
                    tracks = tracker_instance.update(detections, frame)
                else:
                    tracks = tracker_instance.update(np.empty((0, 6)), frame)

                if len(tracks) > 0:
                    for track in tracks:
                        x1, y1, x2, y2, track_id, conf, cls, idx = track
                        all_tracks_for_seq.append(
                            f"{frame_num + 1},{int(track_id)},{x1:.2f},{y1:.2f},{x2-x1:.2f},{y2-y1:.2f},{conf:.6f},-1,-1,-1\n"
                        )

            with open(output_txt_path, 'w') as f:
                f.writelines(all_tracks_for_seq)

        print(f"-> Teste ByteTrack com Confiança {conf_thresh} concluído e salvo no Drive!")

if __name__ == '__main__':
    run_fase1_bytetrack()
    print("\nTODOS OS TESTES DA FASE 1 DO BYTETRACK TERMINADOS!")

### 4.4. DeepSORT — preparação da biblioteca

In [ ]:
import os
import shutil
import glob
from google.colab import drive

DRIVE_ZIP_PATH = '/content/drive/MyDrive/SoccerNet/bibliotecas_git/deep_sort_realtime-master.zip'

shutil.copy(DRIVE_ZIP_PATH, '/content/library.zip')

#DESCOMPACTANDO A BIBLIOTECA
!unzip -o -q /content/library.zip -d /content/
os.remove('/content/library.zip')

unzipped_folder_list = glob.glob('/content/deep_sort_*-master/')
unzipped_folder_path = unzipped_folder_list[0]
%cd {unzipped_folder_path}
!pip install . #instala a partir da pasta local
%cd /content/



In [ ]:
import os
import cv2
import torch
import numpy as np
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort

# 1. DIRETÓRIOS DA VALIDAÇÃO (Blindado contra vazamento de dados!)
BASE_INPUT_DIR = '/content/drive/MyDrive/SoccerNet/tracking/train/'
YOLO_MODEL_PATH = '/content/drive/MyDrive/SoccerNet/runs/treino_cego_albumentations/weights/best.pt'
BASE_OUTPUT_DIR = '/content/drive/MyDrive/SoccerNet/tracking_fase1_deepsort/' # Pasta exclusiva

# 2. A SUA LISTA DE VALIDAÇÃO DOS 12 VÍDEOS
sequence_folders = [
    "SNMOT-113", "SNMOT-157", "SNMOT-066", "SNMOT-167",
    "SNMOT-068", "SNMOT-074", "SNMOT-075", "SNMOT-077",
    "SNMOT-161", "SNMOT-061", "SNMOT-067", "SNMOT-154"
]

# Os três limiares do YOLO que vamos testar
lista_confidences = [0.05, 0.3, 0.5]

# Parâmetros PADRÃO do DeepSORT (Baseados na sua Tabela 1)
DEEPSORT_MAX_AGE = 30
DEEPSORT_N_INIT = 3

def run_fase1_deepsort():
    DEVICE = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')
    print("Carregando o modelo YOLO best.pt...")
    yolo_model = YOLO(YOLO_MODEL_PATH)
    yolo_model.to(DEVICE)

    # LOOP 1: Testa cada limiar de confiança do YOLO
    for conf_thresh in lista_confidences:

        tracker_name = f'DeepSORT_Padrao_thr{conf_thresh}'
        output_txt_dir = os.path.join(BASE_OUTPUT_DIR, tracker_name, 'txt_files')
        os.makedirs(output_txt_dir, exist_ok=True)

        print(f"\n=======================================================")
        print(f"FASE 1 - TESTANDO CONFIANÇA YOLO COM DEEPSORT: {conf_thresh}")
        print(f"=======================================================\n")

        # LOOP 2: Processa APENAS os 12 vídeos de validação
        for seq_folder_name in sequence_folders:
            print(f"--- Processando Sequência: {seq_folder_name} ---")

            # Instancia o DeepSORT com os valores padrão congelados
            deepsort_tracker = DeepSort(max_age=DEEPSORT_MAX_AGE, n_init=DEEPSORT_N_INIT)

            current_image_dir = os.path.join(BASE_INPUT_DIR, seq_folder_name, 'img1')
            output_txt_path = os.path.join(output_txt_dir, f"{seq_folder_name}.txt")

            if not os.path.exists(current_image_dir):
                print(f"ERRO: Pasta {current_image_dir} não encontrada. Pulando...")
                continue

            image_files = sorted([f for f in os.listdir(current_image_dir) if f.lower().endswith('.jpg')])
            if not image_files: continue

            all_tracks_for_seq = []

            for frame_num, image_name in enumerate(image_files):
                # Imprime progresso
                if (frame_num + 1) % 200 == 0:
                    print(f"   ...processando frame {frame_num + 1}/{len(image_files)}")

                frame = cv2.imread(os.path.join(current_image_dir, image_name))
                if frame is None: continue

                # YOLO detecta usando o limiar atual da rodada
                yolo_results = yolo_model.predict(frame, device=DEVICE, verbose=False, conf=conf_thresh)

                # Formata as detecções do jeito que o DeepSORT exige
                detections_for_tracker = []
                for box in yolo_results[0].boxes:
                    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                    conf = box.conf[0].cpu().item()
                    class_name = yolo_model.names[int(box.cls[0].cpu().item())]
                    bbox_xywh = [x1, y1, (x2 - x1), (y2 - y1)]
                    detections_for_tracker.append((bbox_xywh, conf, class_name))

                # Alimenta o tracker. Se estiver vazio, o DeepSORT lida bem com a lista vazia
                tracks = deepsort_tracker.update_tracks(detections_for_tracker, frame=frame)

                for track in tracks:
                    # Só salva se o tracker tiver certeza de que é um objeto real
                    if not track.is_confirmed():
                        continue

                    track_id = track.track_id
                    ltrb = track.to_tlbr()
                    x1, y1, x2, y2 = ltrb
                    track_conf = track.get_det_conf()
                    if track_conf is None: track_conf = -1

                    all_tracks_for_seq.append(
                        f"{frame_num + 1},{track_id},{x1:.2f},{y1:.2f},{x2-x1:.2f},{y2-y1:.2f},{track_conf:.6f},-1,-1,-1\n"
                    )

            with open(output_txt_path, 'w') as f:
                f.writelines(all_tracks_for_seq)

        print(f"-> Teste DeepSORT com Confiança {conf_thresh} concluído e salvo no Drive!")

if __name__ == '__main__':
    run_fase1_deepsort()
    print("\nTODOS OS TESTES DA FASE 1 DO DEEPSORT TERMINADOS!")

### 4.5. Avaliação da Fase 1

In [ ]:
import motmetrics as mm
import os
import pandas as pd
import numpy as np

if not hasattr(np, 'asfarray'):
    np.asfarray = lambda a, *args, **kwargs: np.asarray(a, dtype=float)

# 1. DIRETÓRIO DO GABARITO (Onde ficam os vídeos que usamos na validação)
GT_DIR = '/content/drive/MyDrive/SoccerNet/tracking/train/'

# 2. A LISTA DE VALIDAÇÃO (Trava de Segurança!)
sequence_folders = [
    "SNMOT-113", "SNMOT-157", "SNMOT-066", "SNMOT-167",
    "SNMOT-068", "SNMOT-074", "SNMOT-075", "SNMOT-077",
    "SNMOT-161", "SNMOT-061", "SNMOT-067", "SNMOT-154"
]

# 3. MAPEAMENTO DOS RESULTADOS DA FASE 1
# Formato: 'Nome na Tabela' : 'Caminho exato da pasta gerada'
TRACKERS_TO_EVALUATE = {
    'OcSORT_thr0.05': '/content/drive/MyDrive/SoccerNet/tracking_fase1_confianca/OcSORT_Padrao_thr0.05',
    'OcSORT_thr0.3':  '/content/drive/MyDrive/SoccerNet/tracking_fase1_confianca/OcSORT_Padrao_thr0.3',
    'OcSORT_thr0.5':  '/content/drive/MyDrive/SoccerNet/tracking_fase1_confianca/OcSORT_Padrao_thr0.5',

    'StrongSORT_thr0.05': '/content/drive/MyDrive/SoccerNet/tracking_fase1_strongsort/StrongSORT_Padrao_thr0.05',
    'StrongSORT_thr0.3':  '/content/drive/MyDrive/SoccerNet/tracking_fase1_strongsort/StrongSORT_Padrao_thr0.3',
    'StrongSORT_thr0.5':  '/content/drive/MyDrive/SoccerNet/tracking_fase1_strongsort/StrongSORT_Padrao_thr0.5',

    'ByteTrack_thr0.05': '/content/drive/MyDrive/SoccerNet/tracking_fase1_bytetrack/ByteTrack_Padrao_thr0.05',
    'ByteTrack_thr0.3':  '/content/drive/MyDrive/SoccerNet/tracking_fase1_bytetrack/ByteTrack_Padrao_thr0.3',
    'ByteTrack_thr0.5':  '/content/drive/MyDrive/SoccerNet/tracking_fase1_bytetrack/ByteTrack_Padrao_thr0.5',

    'DeepSORT_thr0.05': '/content/drive/MyDrive/SoccerNet/tracking_fase1_deepsort/DeepSORT_Padrao_thr0.05',
    'DeepSORT_thr0.3':  '/content/drive/MyDrive/SoccerNet/tracking_fase1_deepsort/DeepSORT_Padrao_thr0.3',
    'DeepSORT_thr0.5':  '/content/drive/MyDrive/SoccerNet/tracking_fase1_deepsort/DeepSORT_Padrao_thr0.5',
}

# Cria os acumuladores para cada tracker
accs = {name: mm.MOTAccumulator(auto_id=False) for name in TRACKERS_TO_EVALUATE.keys()}

print("Lendo arquivos de gabarito e resultados...")

# Inicia o loop travado APENAS nos 12 vídeos de validação
for seq_name in sequence_folders:
    gt_file_path = os.path.join(GT_DIR, seq_name, 'gt', 'gt.txt')
    if not os.path.exists(gt_file_path):
        print(f"Aviso: Gabarito de {seq_name} não encontrado!")
        continue

    gt = mm.io.loadtxt(gt_file_path, fmt='mot15-2D').sort_index()

    for tracker_name, tracker_base_dir in TRACKERS_TO_EVALUATE.items():
        ts_file_path = os.path.join(tracker_base_dir, 'txt_files', f'{seq_name}.txt')

        if not os.path.exists(ts_file_path):
            continue # Se o teste ainda não rodou, ele pula silenciosamente

        ts = mm.io.loadtxt(ts_file_path, fmt='mot15-2D').sort_index()

        frame_ids_gt = gt.index.get_level_values('FrameId').unique()
        frame_ids_ts = ts.index.get_level_values('FrameId').unique()
        all_frame_ids = sorted(list(set(frame_ids_gt) | set(frame_ids_ts)))

        for frame_id in all_frame_ids:
            gt_for_frame = gt.loc[frame_id] if frame_id in frame_ids_gt else pd.DataFrame()
            ts_for_frame = ts.loc[frame_id] if frame_id in frame_ids_ts else pd.DataFrame()

            if ts_for_frame.empty:
                dists = np.empty((len(gt_for_frame), 0))
            else:
                dists = mm.distances.iou_matrix(
                    gt_for_frame[['X', 'Y', 'Width', 'Height']],
                    ts_for_frame[['X', 'Y', 'Width', 'Height']],
                    max_iou=0.5
                )

            accs[tracker_name].update(gt_for_frame.index, ts_for_frame.index, dists, frameid=frame_id)

mh = mm.metrics.create()
all_summaries = []

print("\nProcessando métricas (MOTA, FP, FN, IDs)...")
for tracker_name, acc in accs.items():
    if not acc.events.empty:
        acc.events.sort_index(inplace=True)
        summary = mh.compute(acc, metrics=mm.metrics.motchallenge_metrics, name=tracker_name)
        all_summaries.append(summary)

if not all_summaries:
    print("\nNenhum resultado de tracker foi encontrado. Verifique se os treinamentos terminaram.")
else:
    summary_final = pd.concat(all_summaries)

    # As colunas exatas da sua Tabela 2 do TCC
    cols_to_show = ['mota', 'num_false_positives', 'num_misses', 'num_switches']
    summary_filtered = summary_final[[col for col in cols_to_show if col in summary_final.columns]]

    # Renomeando as colunas para ficar idêntico ao seu TCC
    summary_filtered = summary_filtered.rename(columns={
        'mota': 'MOTA (%)',
        'num_false_positives': 'FP',
        'num_misses': 'FN',
        'num_switches': 'IDs'
    })

    # MOTA é uma proporção no motmetrics, vamos multiplicar por 100 para virar porcentagem
    if 'MOTA (%)' in summary_filtered.columns:
        summary_filtered['MOTA (%)'] = summary_filtered['MOTA (%)'] * 100

    print("\n\n=======================================================")
    print("PLACAR FINAL - FASE 1: OTIMIZAÇÃO DO LIMIAR DE CONFIANÇA")
    print("=======================================================\n")
    pd.set_option('display.float_format', '{:.1f}'.format)
    print(summary_filtered)

Lendo arquivos de gabarito e resultados...

Processando métricas (MOTA, FP, FN, IDs)...


PLACAR FINAL - FASE 1: OTIMIZAÇÃO DO LIMIAR DE CONFIANÇA

                    MOTA (%)      FP     FN   IDs
OcSORT_thr0.05          62.0   30547  23598  7851
OcSORT_thr0.3           80.0    5769  24603  2149
OcSORT_thr0.5           77.2    2517  33444  1164
StrongSORT_thr0.05      64.9   33930  17390  5903
StrongSORT_thr0.3       80.4    8282  21339  2329
StrongSORT_thr0.5       79.2    3126  29480  1262
ByteTrack_thr0.05       79.6    3304  28818  1110
ByteTrack_thr0.3        79.4    2759  29816  1057
ByteTrack_thr0.5        76.4    2149  35075  1170
DeepSORT_thr0.05       -43.4  214192  14439  5100
DeepSORT_thr0.3         60.4   43569  19293  1640
DeepSORT_thr0.5         69.7   23868  24371  1109


## 5. Rastreamento — Fase 2 (sintonia fina dos hiperparâmetros)

### 5.1. OC-SORT

In [ ]:
import os
import cv2
import torch
import numpy as np
from ultralytics import YOLO
from boxmot import OcSort

# 1. DIRETÓRIOS DA VALIDAÇÃO
BASE_INPUT_DIR = '/content/drive/MyDrive/SoccerNet/tracking/train/'
YOLO_MODEL_PATH = '/content/drive/MyDrive/SoccerNet/runs/treino_cego_albumentations/weights/best.pt'
BASE_OUTPUT_DIR = '/content/drive/MyDrive/SoccerNet/tracking_fase2_ocsort/'

sequence_folders = [
    "SNMOT-113", "SNMOT-157", "SNMOT-066", "SNMOT-167",
    "SNMOT-068", "SNMOT-074", "SNMOT-075", "SNMOT-077",
    "SNMOT-161", "SNMOT-061", "SNMOT-067", "SNMOT-154"
]

# 2. O CAMPEÃO DA FASE 1 TRANCADO
CONF_THRESH = 0.3

# 3. AS CONFIGURAÇÕES DA FASE 2 (Sua imagem + 1 teste extra)
# Formato: (max_age, n_init, asso_threshold)
configs_fase2_ocsort = [
    (30, 3, 0.2), # Teste 1 da sua imagem
    (30, 3, 0.4), # Teste 2 da sua imagem
    (50, 3, 0.2), # Teste 3 da sua imagem
    (30, 5, 0.3)  # Teste 4 EXTRA: Isolando apenas o impacto de um n_init mais rigoroso
]

def run_fase2_ocsort():
    DEVICE = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')
    print("Carregando YOLO para a Fase 2...")
    yolo_model = YOLO(YOLO_MODEL_PATH)
    yolo_model.to(DEVICE)

    for age, init, asso in configs_fase2_ocsort:
        tracker_name = f'OcSORT_age{age}_init{init}_asso{asso}'
        output_txt_dir = os.path.join(BASE_OUTPUT_DIR, tracker_name, 'txt_files')
        os.makedirs(output_txt_dir, exist_ok=True)

        print(f"\n=======================================================")
        print(f"FASE 2 - OCSORT: Max_Age={age} | N_Init={init} | Asso={asso}")
        print(f"=======================================================\n")

        for seq_folder_name in sequence_folders:
            print(f"--- Processando Sequência: {seq_folder_name} ---")

            # Instancia o Tracker com os hiperparâmetros da rodada
            tracker_instance = OcSort(
                det_thresh=CONF_THRESH,
                max_age=age,
                min_hits=init, # min_hits é o nome do n_init no OcSORT
                asso_threshold=asso
            )

            current_image_dir = os.path.join(BASE_INPUT_DIR, seq_folder_name, 'img1')
            output_txt_path = os.path.join(output_txt_dir, f"{seq_folder_name}.txt")

            image_files = sorted([f for f in os.listdir(current_image_dir) if f.lower().endswith('.jpg')])
            if not image_files: continue

            all_tracks_for_seq = []

            for frame_num, image_name in enumerate(image_files):
                if (frame_num + 1) % 400 == 0:
                    print(f"   ...frame {frame_num + 1}/{len(image_files)}")

                frame = cv2.imread(os.path.join(current_image_dir, image_name))
                if frame is None: continue

                # YOLO Fixo no 0.3
                yolo_results = yolo_model.predict(frame, device=DEVICE, verbose=False, conf=CONF_THRESH)
                detections = yolo_results[0].boxes.data.cpu().numpy()

                if detections.shape[0] > 0:
                    tracks = tracker_instance.update(detections, frame)
                else:
                    tracks = tracker_instance.update(np.empty((0, 6)), frame)

                if len(tracks) > 0:
                    for track in tracks:
                        x1, y1, x2, y2, track_id, conf, cls, idx = track
                        all_tracks_for_seq.append(
                            f"{frame_num + 1},{int(track_id)},{x1:.2f},{y1:.2f},{x2-x1:.2f},{y2-y1:.2f},{conf:.6f},-1,-1,-1\n"
                        )

            with open(output_txt_path, 'w') as f:
                f.writelines(all_tracks_for_seq)

        print(f"-> Teste {tracker_name} concluído!")

if __name__ == '__main__':
    run_fase2_ocsort()
    print("\nFASE 2 DO OCSORT FINALIZADA!")

In [ ]:
import os
import cv2
import torch
import numpy as np
from pathlib import Path
from ultralytics import YOLO
from boxmot import StrongSort

# 1. DIRETÓRIOS DA VALIDAÇÃO
BASE_INPUT_DIR = '/content/drive/MyDrive/SoccerNet/tracking/train/'
YOLO_MODEL_PATH = '/content/drive/MyDrive/SoccerNet/runs/treino_cego_albumentations/weights/best.pt'
BASE_OUTPUT_DIR = '/content/drive/MyDrive/SoccerNet/tracking_fase2_strongsort/'
REID_MODEL_PATH = 'osnet_x0_25_msmt17.pt'

sequence_folders = [
    "SNMOT-113", "SNMOT-157", "SNMOT-066", "SNMOT-167",
    "SNMOT-068", "SNMOT-074", "SNMOT-075", "SNMOT-077",
    "SNMOT-161", "SNMOT-061", "SNMOT-067", "SNMOT-154"
]

# 2. O CAMPEÃO DA FASE 1 TRANCADO (O porteiro oficial)
CONF_THRESH = 0.3

# 3. AS CONFIGURAÇÕES DA FASE 2
# Formato: (max_age, n_init, max_cos_dist, ema_alpha)
configs_fase2_strongsort = [
    # --- Os seus 3 testes originais (Padrão: ema=0.9) ---
    (120, 3, 0.2, 0.9), # Teste 1: Memória gigante
    (70,  5, 0.1, 0.9), # Teste 2: Super rigoroso na criação e na distância visual
    (90,  5, 0.2, 0.9), # Teste 3: Equilíbrio

    # --- O Teste Avançado para a Discussão ---
    # Teste 4: Memória visual "lenta". Demora mais para esquecer a aparência antiga do jogador.
    (90,  3, 0.2, 0.5)
]

def run_fase2_strongsort():
    DEVICE = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')
    print("Carregando o YOLO para a Fase 2...")
    yolo_model = YOLO(YOLO_MODEL_PATH)
    yolo_model.to(DEVICE)

    for age, init, cos, ema in configs_fase2_strongsort:

        tracker_name = f'StrongSORT_age{age}_init{init}_cos{cos}_ema{ema}'
        output_txt_dir = os.path.join(BASE_OUTPUT_DIR, tracker_name, 'txt_files')
        os.makedirs(output_txt_dir, exist_ok=True)

        print(f"\n=======================================================")
        print(f"FASE 2 - STRONGSORT: {tracker_name}")
        print(f"=======================================================\n")

        for seq_folder_name in sequence_folders:
            print(f"--- Processando a sequência: {seq_folder_name} ---")

            # Instancia o tracker com os hiperparâmetros da rodada
            tracker_instance = StrongSort(
                reid_weights=Path(REID_MODEL_PATH),
                device=DEVICE,
                half=False,
                max_age=age,
                n_init=init,
                max_cos_dist=cos,
                ema_alpha=ema
            )

            current_image_dir = os.path.join(BASE_INPUT_DIR, seq_folder_name, 'img1')
            output_txt_path = os.path.join(output_txt_dir, f"{seq_folder_name}.txt")

            if not os.path.exists(current_image_dir):
                print(f"ERRO: Pasta {current_image_dir} não encontrada. Pulando...")
                continue

            image_files = sorted([f for f in os.listdir(current_image_dir) if f.lower().endswith('.jpg')])
            if not image_files: continue

            all_tracks_for_seq = []

            for frame_num, image_name in enumerate(image_files):
                if (frame_num + 1) % 400 == 0:
                    print(f"   ...frame {frame_num + 1}/{len(image_files)}")

                frame = cv2.imread(os.path.join(current_image_dir, image_name))
                if frame is None: continue

                # YOLO detecta SEMPRE com o 0.3
                yolo_results = yolo_model.predict(frame, device=DEVICE, verbose=False, conf=CONF_THRESH)
                detections = yolo_results[0].boxes.data.cpu().numpy()

                if detections.shape[0] > 0:
                    tracks = tracker_instance.update(detections, frame)
                else:
                    tracks = tracker_instance.update(np.empty((0, 6)), frame)

                if len(tracks) > 0:
                    for track in tracks:
                        x1, y1, x2, y2, track_id, conf, cls, idx = track
                        all_tracks_for_seq.append(
                            f"{frame_num + 1},{int(track_id)},{x1:.2f},{y1:.2f},{x2-x1:.2f},{y2-y1:.2f},{conf:.6f},-1,-1,-1\n"
                        )

            with open(output_txt_path, 'w') as f:
                f.writelines(all_tracks_for_seq)

        print(f"-> Teste {tracker_name} concluído!")

if __name__ == '__main__':
    run_fase2_strongsort()
    print("\nFASE 2 DO STRONGSORT FINALIZADA!")

### 5.2. ByteTrack

In [ ]:
import os
import cv2
import torch
import numpy as np
from ultralytics import YOLO
from boxmot import ByteTrack

# 1. DIRETÓRIOS DA VALIDAÇÃO (O cofre)
BASE_INPUT_DIR = '/content/drive/MyDrive/SoccerNet/tracking/train/'
YOLO_MODEL_PATH = '/content/drive/MyDrive/SoccerNet/runs/treino_cego_albumentations/weights/best.pt'
BASE_OUTPUT_DIR = '/content/drive/MyDrive/SoccerNet/tracking_fase2_bytetrack/'

sequence_folders = [
    "SNMOT-113", "SNMOT-157", "SNMOT-066", "SNMOT-167",
    "SNMOT-068", "SNMOT-074", "SNMOT-075", "SNMOT-077",
    "SNMOT-161", "SNMOT-061", "SNMOT-067", "SNMOT-154"
]

# 2. O CAMPEÃO DA FASE 1 TRANCADO (A Mágica do ByteTrack)
CONF_THRESH = 0.05

# 3. AS CONFIGURAÇÕES DA FASE 2
# Formato: (track_buffer/age, track_thresh/hi, match_thresh, track_low_thresh)
configs_fase2_bytetrack = [
    # --- Os seus 3 testes originais (Padrão: low_thresh=0.1) ---
    (30, 0.7, 0.9, 0.1),  # Teste 1: Memória curta, super rigoroso na criação
    (45, 0.5, 0.7, 0.1),  # Teste 2: Equilibrado
    (60, 0.6, 0.8, 0.1),  # Teste 3: Memória longa

    # --- O Teste 4 (A sua ideia) ---
    # Subindo o Low Thresh (Cortando o fundo do poço para ver se limpa Falsos Positivos)
    (30, 0.6, 0.8, 0.2)
]

def run_fase2_bytetrack():
    DEVICE = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')
    print("Carregando o YOLO para a Fase 2...")
    yolo_model = YOLO(YOLO_MODEL_PATH)
    yolo_model.to(DEVICE)

    for age, hi_thr, match_thr, low_thr in configs_fase2_bytetrack:

        tracker_name = f'ByteTrack_age{age}_hi{hi_thr}_match{match_thr}_low{low_thr}'
        output_txt_dir = os.path.join(BASE_OUTPUT_DIR, tracker_name, 'txt_files')
        os.makedirs(output_txt_dir, exist_ok=True)

        print(f"\n=======================================================")
        print(f"FASE 2 - BYTETRACK: Age={age} | Hi={hi_thr} | Match={match_thr} | Low={low_thr}")
        print(f"=======================================================\n")

        for seq_folder_name in sequence_folders:
            print(f"--- Processando Sequência: {seq_folder_name} ---")

            # Instancia o Tracker
            tracker_instance = ByteTrack(
                track_buffer=age,
                track_thresh=hi_thr,
                match_thresh=match_thr,
                track_low_thresh=low_thr # A sua ideia aplicada aqui!
            )

            current_image_dir = os.path.join(BASE_INPUT_DIR, seq_folder_name, 'img1')
            output_txt_path = os.path.join(output_txt_dir, f"{seq_folder_name}.txt")

            if not os.path.exists(current_image_dir):
                print(f"ERRO: Pasta {current_image_dir} não encontrada. Pulando...")
                continue

            image_files = sorted([f for f in os.listdir(current_image_dir) if f.lower().endswith('.jpg')])
            if not image_files: continue

            all_tracks_for_seq = []

            for frame_num, image_name in enumerate(image_files):
                if (frame_num + 1) % 400 == 0:
                    print(f"   ...frame {frame_num + 1}/{len(image_files)}")

                frame = cv2.imread(os.path.join(current_image_dir, image_name))
                if frame is None: continue

                # YOLO Fixo no 0.05
                yolo_results = yolo_model.predict(frame, device=DEVICE, verbose=False, conf=CONF_THRESH)
                detections = yolo_results[0].boxes.data.cpu().numpy()

                if detections.shape[0] > 0:
                    tracks = tracker_instance.update(detections, frame)
                else:
                    tracks = tracker_instance.update(np.empty((0, 6)), frame)

                if len(tracks) > 0:
                    for track in tracks:
                        x1, y1, x2, y2, track_id, conf, cls, idx = track
                        all_tracks_for_seq.append(
                            f"{frame_num + 1},{int(track_id)},{x1:.2f},{y1:.2f},{x2-x1:.2f},{y2-y1:.2f},{conf:.6f},-1,-1,-1\n"
                        )

            with open(output_txt_path, 'w') as f:
                f.writelines(all_tracks_for_seq)

        print(f"-> Teste {tracker_name} concluído!")

if __name__ == '__main__':
    run_fase2_bytetrack()
    print("\nFASE 2 DO BYTETRACK FINALIZADA!")

### 5.3. DeepSORT

In [ ]:
import os
import cv2
import torch
import numpy as np
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort

# 1. DIRETÓRIOS DA VALIDAÇÃO (Cofre fechado!)
BASE_INPUT_DIR = '/content/drive/MyDrive/SoccerNet/tracking/train/'
YOLO_MODEL_PATH = '/content/drive/MyDrive/SoccerNet/runs/treino_cego_albumentations/weights/best.pt'
BASE_OUTPUT_DIR = '/content/drive/MyDrive/SoccerNet/tracking_fase2_deepsort/' # Nova pasta para a Fase 2

# 2. A SUA LISTA DE VALIDAÇÃO DOS 12 VÍDEOS
sequence_folders = [
    "SNMOT-113", "SNMOT-157", "SNMOT-066", "SNMOT-167",
    "SNMOT-068", "SNMOT-074", "SNMOT-075", "SNMOT-077",
    "SNMOT-161", "SNMOT-061", "SNMOT-067", "SNMOT-154"
]

# 3. O CAMPEÃO DA FASE 1 TRANCADO
CONF_THRESH = 0.5

# 4. AS CONFIGURAÇÕES DA FASE 2
# Formato: (max_age, n_init, max_cosine_distance)
configs_fase2_deepsort = [
    # --- Os seus 3 testes originais baseados em memória e rigor de criação ---
    (30, 7, 0.2), # Teste 1: Memória curta, absurdamente difícil de iniciar um rastro
    (70, 5, 0.2), # Teste 2: Equilibrado
    (90, 3, 0.2), # Teste 3: Memória longa, fácil de iniciar

    # --- O Teste 4 Extra (O Rigor Visual) ---
    # Usamos o n_init normal, mas cortamos o cosine_distance pela metade (0.1).
    # O tracker vai exigir que o jogador seja visualmente "idêntico" ao frame anterior.
    (30, 3, 0.1)
]

def run_fase2_deepsort():
    DEVICE = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')
    print("Carregando o modelo YOLO best.pt para a Fase 2...")
    yolo_model = YOLO(YOLO_MODEL_PATH)
    yolo_model.to(DEVICE)

    for age, init, cos_dist in configs_fase2_deepsort:

        tracker_name = f'DeepSORT_age{age}_init{init}_cos{cos_dist}'
        output_txt_dir = os.path.join(BASE_OUTPUT_DIR, tracker_name, 'txt_files')
        os.makedirs(output_txt_dir, exist_ok=True)

        print(f"\n=======================================================")
        print(f"FASE 2 - DEEPSORT: Age={age} | Init={init} | Cos_Dist={cos_dist}")
        print(f"=======================================================\n")

        for seq_folder_name in sequence_folders:
            print(f"--- Processando Sequência: {seq_folder_name} ---")

            # Instancia o DeepSORT com os parâmetros da rodada
            deepsort_tracker = DeepSort(
                max_age=age,
                n_init=init,
                max_cosine_distance=cos_dist
            )

            current_image_dir = os.path.join(BASE_INPUT_DIR, seq_folder_name, 'img1')
            output_txt_path = os.path.join(output_txt_dir, f"{seq_folder_name}.txt")

            if not os.path.exists(current_image_dir):
                print(f"ERRO: Pasta {current_image_dir} não encontrada. Pulando...")
                continue

            image_files = sorted([f for f in os.listdir(current_image_dir) if f.lower().endswith('.jpg')])
            if not image_files: continue

            all_tracks_for_seq = []

            for frame_num, image_name in enumerate(image_files):
                if (frame_num + 1) % 400 == 0:
                    print(f"   ...processando frame {frame_num + 1}/{len(image_files)}")

                frame = cv2.imread(os.path.join(current_image_dir, image_name))
                if frame is None: continue

                # YOLO cravado em 0.5
                yolo_results = yolo_model.predict(frame, device=DEVICE, verbose=False, conf=CONF_THRESH)

                detections_for_tracker = []
                for box in yolo_results[0].boxes:
                    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                    conf = box.conf[0].cpu().item()
                    class_name = yolo_model.names[int(box.cls[0].cpu().item())]
                    bbox_xywh = [x1, y1, (x2 - x1), (y2 - y1)]
                    detections_for_tracker.append((bbox_xywh, conf, class_name))

                tracks = deepsort_tracker.update_tracks(detections_for_tracker, frame=frame)

                for track in tracks:
                    if not track.is_confirmed():
                        continue

                    track_id = track.track_id
                    ltrb = track.to_tlbr()
                    x1, y1, x2, y2 = ltrb
                    track_conf = track.get_det_conf()
                    if track_conf is None: track_conf = -1

                    all_tracks_for_seq.append(
                        f"{frame_num + 1},{track_id},{x1:.2f},{y1:.2f},{x2-x1:.2f},{y2-y1:.2f},{track_conf:.6f},-1,-1,-1\n"
                    )

            with open(output_txt_path, 'w') as f:
                f.writelines(all_tracks_for_seq)

        print(f"-> Teste {tracker_name} concluído!")

if __name__ == '__main__':
    run_fase2_deepsort()
    print("\nTODOS OS TESTES DA FASE 2 DO DEEPSORT TERMINADOS!")

### 5.4. Avaliação da Fase 2

In [ ]:
import motmetrics as mm
import os
import pandas as pd
import numpy as np

# === A MÁGICA PARA CONSERTAR O ERRO DO NUMPY 2.0 ===
if not hasattr(np, 'asfarray'):
    np.asfarray = lambda a, *args, **kwargs: np.asarray(a, dtype=float)
# ===================================================

# 1. DIRETÓRIO DO GABARITO (A Validação Blindada)
GT_DIR = '/content/drive/MyDrive/SoccerNet/tracking/train/'

sequence_folders = [
    "SNMOT-113", "SNMOT-157", "SNMOT-066", "SNMOT-167",
    "SNMOT-068", "SNMOT-074", "SNMOT-075", "SNMOT-077",
    "SNMOT-161", "SNMOT-061", "SNMOT-067", "SNMOT-154"
]

# 2. MAPEAMENTO DOS RESULTADOS DA FASE 2
TRACKERS_TO_EVALUATE = {
    # --- OCSORT ---
    'OcSORT_t1_age30_init3_asso0.2': '/content/drive/MyDrive/SoccerNet/tracking_fase2_ocsort/OcSORT_age30_init3_asso0.2',
    'OcSORT_t2_age30_init3_asso0.4': '/content/drive/MyDrive/SoccerNet/tracking_fase2_ocsort/OcSORT_age30_init3_asso0.4',
    'OcSORT_t3_age50_init3_asso0.2': '/content/drive/MyDrive/SoccerNet/tracking_fase2_ocsort/OcSORT_age50_init3_asso0.2',
    'OcSORT_t4_age30_init5_asso0.3': '/content/drive/MyDrive/SoccerNet/tracking_fase2_ocsort/OcSORT_age30_init5_asso0.3',

    # --- STRONGSORT ---
    'StrongSORT_t1_age120_init3_cos0.2_ema0.9': '/content/drive/MyDrive/SoccerNet/tracking_fase2_strongsort/StrongSORT_age120_init3_cos0.2_ema0.9',
    'StrongSORT_t2_age70_init5_cos0.1_ema0.9': '/content/drive/MyDrive/SoccerNet/tracking_fase2_strongsort/StrongSORT_age70_init5_cos0.1_ema0.9',
    'StrongSORT_t3_age90_init5_cos0.2_ema0.9': '/content/drive/MyDrive/SoccerNet/tracking_fase2_strongsort/StrongSORT_age90_init5_cos0.2_ema0.9',
    'StrongSORT_t4_age90_init3_cos0.2_ema0.5': '/content/drive/MyDrive/SoccerNet/tracking_fase2_strongsort/StrongSORT_age90_init3_cos0.2_ema0.5',

    # --- BYTETRACK ---
    'ByteTrack_t1_age30_hi0.7_match0.9_low0.1': '/content/drive/MyDrive/SoccerNet/tracking_fase2_bytetrack/ByteTrack_age30_hi0.7_match0.9_low0.1',
    'ByteTrack_t2_age45_hi0.5_match0.7_low0.1': '/content/drive/MyDrive/SoccerNet/tracking_fase2_bytetrack/ByteTrack_age45_hi0.5_match0.7_low0.1',
    'ByteTrack_t3_age60_hi0.6_match0.8_low0.1': '/content/drive/MyDrive/SoccerNet/tracking_fase2_bytetrack/ByteTrack_age60_hi0.6_match0.8_low0.1',
    'ByteTrack_t4_age30_hi0.6_match0.8_low0.2': '/content/drive/MyDrive/SoccerNet/tracking_fase2_bytetrack/ByteTrack_age30_hi0.6_match0.8_low0.2',

    # --- DEEPSORT ---
    'DeepSORT_t1_age30_init7_cos0.2': '/content/drive/MyDrive/SoccerNet/tracking_fase2_deepsort/DeepSORT_age30_init7_cos0.2',
    'DeepSORT_t2_age70_init5_cos0.2': '/content/drive/MyDrive/SoccerNet/tracking_fase2_deepsort/DeepSORT_age70_init5_cos0.2',
    'DeepSORT_t3_age90_init3_cos0.2': '/content/drive/MyDrive/SoccerNet/tracking_fase2_deepsort/DeepSORT_age90_init3_cos0.2',
    'DeepSORT_t4_age30_init3_cos0.1': '/content/drive/MyDrive/SoccerNet/tracking_fase2_deepsort/DeepSORT_age30_init3_cos0.1',
}

accs = {name: mm.MOTAccumulator(auto_id=False) for name in TRACKERS_TO_EVALUATE.keys()}
print("Lendo arquivos de gabarito e cruzando resultados da Fase 2...")

for seq_name in sequence_folders:
    gt_file_path = os.path.join(GT_DIR, seq_name, 'gt', 'gt.txt')
    if not os.path.exists(gt_file_path): continue

    gt = mm.io.loadtxt(gt_file_path, fmt='mot15-2D').sort_index()

    for tracker_name, tracker_base_dir in TRACKERS_TO_EVALUATE.items():
        ts_file_path = os.path.join(tracker_base_dir, 'txt_files', f'{seq_name}.txt')

        if not os.path.exists(ts_file_path):
            continue # Pula silenciosamente se o tracker ainda não terminou de rodar

        ts = mm.io.loadtxt(ts_file_path, fmt='mot15-2D').sort_index()

        frame_ids_gt = gt.index.get_level_values('FrameId').unique()
        frame_ids_ts = ts.index.get_level_values('FrameId').unique()
        all_frame_ids = sorted(list(set(frame_ids_gt) | set(frame_ids_ts)))

        for frame_id in all_frame_ids:
            gt_for_frame = gt.loc[frame_id] if frame_id in frame_ids_gt else pd.DataFrame()
            ts_for_frame = ts.loc[frame_id] if frame_id in frame_ids_ts else pd.DataFrame()

            if ts_for_frame.empty:
                dists = np.empty((len(gt_for_frame), 0))
            else:
                dists = mm.distances.iou_matrix(
                    gt_for_frame[['X', 'Y', 'Width', 'Height']],
                    ts_for_frame[['X', 'Y', 'Width', 'Height']],
                    max_iou=0.5
                )

            accs[tracker_name].update(gt_for_frame.index, ts_for_frame.index, dists, frameid=frame_id)

mh = mm.metrics.create()
all_summaries = []

print("\nCalculando Placar Final...")
for tracker_name, acc in accs.items():
    if not acc.events.empty:
        acc.events.sort_index(inplace=True)
        summary = mh.compute(acc, metrics=mm.metrics.motchallenge_metrics, name=tracker_name)
        all_summaries.append(summary)

if not all_summaries:
    print("\nNenhum resultado encontrado. Os testes ainda estão rodando?")
else:
    summary_final = pd.concat(all_summaries)

    cols_to_show = ['mota', 'num_false_positives', 'num_misses', 'num_switches']
    summary_filtered = summary_final[[col for col in cols_to_show if col in summary_final.columns]]

    summary_filtered = summary_filtered.rename(columns={
        'mota': 'MOTA (%)',
        'num_false_positives': 'FP',
        'num_misses': 'FN',
        'num_switches': 'IDs'
    })

    if 'MOTA (%)' in summary_filtered.columns:
        summary_filtered['MOTA (%)'] = summary_filtered['MOTA (%)'] * 100

    print("\n\n=======================================================")
    print("PLACAR FINAL - FASE 2: SINTONIA FINA DOS HIPERPARÂMETROS")
    print("=======================================================\n")
    pd.set_option('display.float_format', '{:.1f}'.format)
    print(summary_filtered)

Lendo arquivos de gabarito e cruzando resultados da Fase 2...

Calculando Placar Final...


PLACAR FINAL - FASE 2: SINTONIA FINA DOS HIPERPARÂMETROS

                                          MOTA (%)     FP     FN   IDs
OcSORT_t1_age30_init3_asso0.2                 80.0   5769  24603  2149
OcSORT_t2_age30_init3_asso0.4                 80.0   5769  24603  2149
OcSORT_t3_age50_init3_asso0.2                 79.9   5760  24713  2222
OcSORT_t4_age30_init5_asso0.3                 78.7   4551  28374  1833
StrongSORT_t1_age120_init3_cos0.2_ema0.9      80.0   8587  21247  2696
StrongSORT_t2_age70_init5_cos0.1_ema0.9       80.1   5727  24768  2005
StrongSORT_t3_age90_init5_cos0.2_ema0.9       80.4   7209  22620  2031
StrongSORT_t4_age90_init3_cos0.2_ema0.5       80.1   8684  21175  2622
ByteTrack_t1_age30_hi0.7_match0.9_low0.1      71.1   1854  44257  1001
ByteTrack_t2_age45_hi0.5_match0.7_low0.1      80.4   4623  25469  1909
ByteTrack_t3_age60_hi0.6_match0.8_low0.1      79.6   3319  28801  112

## 6. Rastreamento — Fase 3 (teste cego em ambiente real)

### 6.1. ByteTrack

In [ ]:
import os
import cv2
import torch
import numpy as np
from ultralytics import YOLO
from boxmot import ByteTrack

# ==============================================================================
# FASE 3 - O COFRE ABERTO (TESTE CEGO) - BYTETRACK
# ==============================================================================
BASE_INPUT_DIR = '/content/drive/MyDrive/SoccerNet/tracking/test/'
YOLO_MODEL_PATH = '/content/drive/MyDrive/SoccerNet/runs/treino_cego_albumentations/weights/best.pt'
BASE_OUTPUT_DIR = '/content/drive/MyDrive/SoccerNet/tracking_FASE3_FINAL/'

sequence_folders = sorted([f for f in os.listdir(BASE_INPUT_DIR) if f.startswith("SNMOT-")])

# O CAMPEÃO DO BYTETRACK (Configuração Top 1 da Fase 2)
CONF_THRESH = 0.05
TRACKER_NAME = "ByteTrack_Final_Campeao"

def run_teste_cego_bytetrack():
    DEVICE = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')
    print(f"Iniciando FASE 3 para: {TRACKER_NAME}")

    yolo_model = YOLO(YOLO_MODEL_PATH)
    yolo_model.to(DEVICE)

    output_txt_dir = os.path.join(BASE_OUTPUT_DIR, TRACKER_NAME, 'txt_files')
    os.makedirs(output_txt_dir, exist_ok=True)

    for seq_folder_name in sequence_folders:
        print(f"--- Processando Sequência Inédita: {seq_folder_name} ---")

        # Recriando a instância para zerar a memória a cada vídeo!
        tracker_instance = ByteTrack(
            track_buffer=45,
            track_thresh=0.5,
            match_thresh=0.7,
            track_low_thresh=0.1
        )

        current_image_dir = os.path.join(BASE_INPUT_DIR, seq_folder_name, 'img1')
        output_txt_path = os.path.join(output_txt_dir, f"{seq_folder_name}.txt")

        if not os.path.exists(current_image_dir): continue
        image_files = sorted([f for f in os.listdir(current_image_dir) if f.lower().endswith('.jpg')])
        if not image_files: continue

        all_tracks_for_seq = []

        for frame_num, image_name in enumerate(image_files):
            if (frame_num + 1) % 400 == 0:
                print(f"   ...frame {frame_num + 1}/{len(image_files)}")

            frame = cv2.imread(os.path.join(current_image_dir, image_name))
            if frame is None: continue

            # YOLO detecta
            yolo_results = yolo_model.predict(frame, device=DEVICE, verbose=False, conf=CONF_THRESH)
            detections = yolo_results[0].boxes.data.cpu().numpy()

            if detections.shape[0] > 0:
                tracks = tracker_instance.update(detections, frame)
            else:
                tracks = tracker_instance.update(np.empty((0, 6)), frame)

            if len(tracks) > 0:
                for track in tracks:
                    x1, y1, x2, y2, track_id, conf, cls, idx = track
                    all_tracks_for_seq.append(f"{frame_num + 1},{int(track_id)},{x1:.2f},{y1:.2f},{x2-x1:.2f},{y2-y1:.2f},{conf:.6f},-1,-1,-1\n")

        with open(output_txt_path, 'w') as f:
            f.writelines(all_tracks_for_seq)

    print(f"-> Teste Final do {TRACKER_NAME} concluído com sucesso! Salvo no Drive.")

if __name__ == '__main__':
    run_teste_cego_bytetrack()

In [ ]:
import os
import cv2
import torch
import numpy as np
from ultralytics import YOLO
from boxmot import OcSort

# ==============================================================================
# FASE 3 - O COFRE ABERTO (TESTE CEGO) - OCSORT
# ==============================================================================
BASE_INPUT_DIR = '/content/drive/MyDrive/SoccerNet/tracking/test/'
YOLO_MODEL_PATH = '/content/drive/MyDrive/SoccerNet/runs/treino_cego_albumentations/weights/best.pt'
BASE_OUTPUT_DIR = '/content/drive/MyDrive/SoccerNet/tracking_FASE3_FINAL/'

# Vai ler TODAS as pastas da base de teste inédita (49 vídeos)
sequence_folders = sorted([f for f in os.listdir(BASE_INPUT_DIR) if f.startswith("SNMOT-")])

# O CAMPEÃO DO OCSORT (Configuração Top 1 da Fase 2)
CONF_THRESH = 0.3
TRACKER_NAME = "OcSORT_Final_Campeao"

def run_teste_cego_ocsort():
    DEVICE = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')
    print(f"Iniciando FASE 3 para: {TRACKER_NAME}")

    yolo_model = YOLO(YOLO_MODEL_PATH)
    yolo_model.to(DEVICE)

    output_txt_dir = os.path.join(BASE_OUTPUT_DIR, TRACKER_NAME, 'txt_files')
    os.makedirs(output_txt_dir, exist_ok=True)

    for seq_folder_name in sequence_folders:
        print(f"--- Processando Sequência Inédita: {seq_folder_name} ---")

        # Recriando a instância para zerar a memória a cada vídeo!
        tracker_instance = OcSort(
            det_thresh=CONF_THRESH,
            max_age=30,
            min_hits=3,
            asso_threshold=0.2
        )

        current_image_dir = os.path.join(BASE_INPUT_DIR, seq_folder_name, 'img1')
        output_txt_path = os.path.join(output_txt_dir, f"{seq_folder_name}.txt")

        if not os.path.exists(current_image_dir): continue
        image_files = sorted([f for f in os.listdir(current_image_dir) if f.lower().endswith('.jpg')])
        if not image_files: continue

        all_tracks_for_seq = []

        for frame_num, image_name in enumerate(image_files):
            # Print de progresso a cada 400 frames
            if (frame_num + 1) % 400 == 0:
                print(f"   ...frame {frame_num + 1}/{len(image_files)}")

            frame = cv2.imread(os.path.join(current_image_dir, image_name))
            if frame is None: continue

            # YOLO detecta
            yolo_results = yolo_model.predict(frame, device=DEVICE, verbose=False, conf=CONF_THRESH)
            detections = yolo_results[0].boxes.data.cpu().numpy()

            if detections.shape[0] > 0:
                tracks = tracker_instance.update(detections, frame)
            else:
                tracks = tracker_instance.update(np.empty((0, 6)), frame)

            if len(tracks) > 0:
                for track in tracks:
                    x1, y1, x2, y2, track_id, conf, cls, idx = track
                    all_tracks_for_seq.append(f"{frame_num + 1},{int(track_id)},{x1:.2f},{y1:.2f},{x2-x1:.2f},{y2-y1:.2f},{conf:.6f},-1,-1,-1\n")

        with open(output_txt_path, 'w') as f:
            f.writelines(all_tracks_for_seq)

    print(f"-> Teste Final do {TRACKER_NAME} concluído com sucesso! Salvo no Drive.")

if __name__ == '__main__':
    run_teste_cego_ocsort()

### 6.2. StrongSORT

In [ ]:
import os
import cv2
import torch
import numpy as np
from pathlib import Path
from ultralytics import YOLO
from boxmot import StrongSort

# ==============================================================================
# FASE 3 - O COFRE ABERTO (TESTE CEGO) - STRONGSORT
# ==============================================================================
BASE_INPUT_DIR = '/content/drive/MyDrive/SoccerNet/tracking/test/'
YOLO_MODEL_PATH = '/content/drive/MyDrive/SoccerNet/runs/treino_cego_albumentations/weights/best.pt'
BASE_OUTPUT_DIR = '/content/drive/MyDrive/SoccerNet/tracking_FASE3_FINAL/'
REID_MODEL_PATH = 'osnet_x0_25_msmt17.pt'

sequence_folders = sorted([f for f in os.listdir(BASE_INPUT_DIR) if f.startswith("SNMOT-")])

CONF_THRESH = 0.3
TRACKER_NAME = "StrongSORT_Final_Campeao"

def run_teste_cego_strongsort():
    DEVICE = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')
    print(f"Iniciando FASE 3 para: {TRACKER_NAME}")

    yolo_model = YOLO(YOLO_MODEL_PATH)
    yolo_model.to(DEVICE)

    output_txt_dir = os.path.join(BASE_OUTPUT_DIR, TRACKER_NAME, 'txt_files')
    os.makedirs(output_txt_dir, exist_ok=True)

    for seq_folder_name in sequence_folders:
        print(f"--- Processando Sequência Inédita: {seq_folder_name} ---")

        # Instância recriada a cada vídeo
        tracker_instance = StrongSort(
            reid_weights=Path(REID_MODEL_PATH),
            device=DEVICE,
            half=False,
            max_age=90,
            n_init=5,
            max_cos_dist=0.2,
            ema_alpha=0.9
        )

        current_image_dir = os.path.join(BASE_INPUT_DIR, seq_folder_name, 'img1')
        output_txt_path = os.path.join(output_txt_dir, f"{seq_folder_name}.txt")

        if not os.path.exists(current_image_dir): continue
        image_files = sorted([f for f in os.listdir(current_image_dir) if f.lower().endswith('.jpg')])
        if not image_files: continue

        all_tracks_for_seq = []

        for frame_num, image_name in enumerate(image_files):
            if (frame_num + 1) % 400 == 0:
                print(f"   ...frame {frame_num + 1}/{len(image_files)}")

            frame = cv2.imread(os.path.join(current_image_dir, image_name))
            if frame is None: continue

            yolo_results = yolo_model.predict(frame, device=DEVICE, verbose=False, conf=CONF_THRESH)
            detections = yolo_results[0].boxes.data.cpu().numpy()

            if detections.shape[0] > 0:
                tracks = tracker_instance.update(detections, frame)
            else:
                tracks = tracker_instance.update(np.empty((0, 6)), frame)

            if len(tracks) > 0:
                for track in tracks:
                    x1, y1, x2, y2, track_id, conf, cls, idx = track
                    all_tracks_for_seq.append(f"{frame_num + 1},{int(track_id)},{x1:.2f},{y1:.2f},{x2-x1:.2f},{y2-y1:.2f},{conf:.6f},-1,-1,-1\n")

        with open(output_txt_path, 'w') as f:
            f.writelines(all_tracks_for_seq)

    print(f"-> Teste Final do {TRACKER_NAME} concluído com sucesso! Salvo no Drive.")

if __name__ == '__main__':
    run_teste_cego_strongsort()

### 6.3. DeepSORT

In [ ]:
import os
import cv2
import torch
import numpy as np
from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort

# ==============================================================================
# FASE 3 - O COFRE ABERTO (TESTE CEGO) - DEEPSORT
# ==============================================================================
BASE_INPUT_DIR = '/content/drive/MyDrive/SoccerNet/tracking/test/'
YOLO_MODEL_PATH = '/content/drive/MyDrive/SoccerNet/runs/treino_cego_albumentations/weights/best.pt'
BASE_OUTPUT_DIR = '/content/drive/MyDrive/SoccerNet/tracking_FASE3_FINAL/'

sequence_folders = sorted([f for f in os.listdir(BASE_INPUT_DIR) if f.startswith("SNMOT-")])

CONF_THRESH = 0.5
TRACKER_NAME = "DeepSORT_Final_Campeao"

def run_teste_cego_deepsort():
    DEVICE = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')
    print(f"Iniciando FASE 3 para: {TRACKER_NAME}")

    yolo_model = YOLO(YOLO_MODEL_PATH)
    yolo_model.to(DEVICE)

    output_txt_dir = os.path.join(BASE_OUTPUT_DIR, TRACKER_NAME, 'txt_files')
    os.makedirs(output_txt_dir, exist_ok=True)

    for seq_folder_name in sequence_folders:
        print(f"--- Processando Sequência Inédita: {seq_folder_name} ---")

        # Instância recriada a cada vídeo
        tracker_instance = DeepSort(
            max_age=30,
            n_init=7,
            max_cosine_distance=0.2
        )

        current_image_dir = os.path.join(BASE_INPUT_DIR, seq_folder_name, 'img1')
        output_txt_path = os.path.join(output_txt_dir, f"{seq_folder_name}.txt")

        if not os.path.exists(current_image_dir): continue
        image_files = sorted([f for f in os.listdir(current_image_dir) if f.lower().endswith('.jpg')])
        if not image_files: continue

        all_tracks_for_seq = []

        for frame_num, image_name in enumerate(image_files):
            if (frame_num + 1) % 400 == 0:
                print(f"   ...frame {frame_num + 1}/{len(image_files)}")

            frame = cv2.imread(os.path.join(current_image_dir, image_name))
            if frame is None: continue

            yolo_results = yolo_model.predict(frame, device=DEVICE, verbose=False, conf=CONF_THRESH)

            detections_for_tracker = []
            for box in yolo_results[0].boxes:
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                conf = box.conf[0].cpu().item()
                class_name = yolo_model.names[int(box.cls[0].cpu().item())]
                bbox_xywh = [x1, y1, (x2 - x1), (y2 - y1)]
                detections_for_tracker.append((bbox_xywh, conf, class_name))

            tracks = tracker_instance.update_tracks(detections_for_tracker, frame=frame)

            for track in tracks:
                if not track.is_confirmed(): continue
                track_id = track.track_id
                ltrb = track.to_tlbr()
                x1, y1, x2, y2 = ltrb
                track_conf = track.get_det_conf()
                if track_conf is None: track_conf = -1
                all_tracks_for_seq.append(f"{frame_num + 1},{track_id},{x1:.2f},{y1:.2f},{x2-x1:.2f},{y2-y1:.2f},{track_conf:.6f},-1,-1,-1\n")

        with open(output_txt_path, 'w') as f:
            f.writelines(all_tracks_for_seq)

    print(f"-> Teste Final do {TRACKER_NAME} concluído com sucesso! Salvo no Drive.")

if __name__ == '__main__':
    run_teste_cego_deepsort()

### 6.4. Avaliação final (placar do TCC)

In [ ]:
import motmetrics as mm
import os
import pandas as pd
import numpy as np

# === VACINA NUMPY 2.0 ===
if not hasattr(np, 'asfarray'):
    np.asfarray = lambda a, *args, **kwargs: np.asarray(a, dtype=float)

# 1. DIRETÓRIO DO GABARITO (O COFRE FINAL)
GT_DIR = '/content/drive/MyDrive/SoccerNet/tracking/test/'

# Lê todas as pastas da base de teste
sequence_folders = sorted([f for f in os.listdir(GT_DIR) if f.startswith("SNMOT-")])

# 2. MAPEAMENTO DOS RESULTADOS FINAIS
TRACKERS_TO_EVALUATE = {
    '1_OcSORT_Campeao': '/content/drive/MyDrive/SoccerNet/tracking_FASE3_FINAL/OcSORT_Final_Campeao',
    '2_ByteTrack_Campeao': '/content/drive/MyDrive/SoccerNet/tracking_FASE3_FINAL/ByteTrack_Final_Campeao',
    '3_StrongSORT_Campeao': '/content/drive/MyDrive/SoccerNet/tracking_FASE3_FINAL/StrongSORT_Final_Campeao',
    '4_DeepSORT_Campeao': '/content/drive/MyDrive/SoccerNet/tracking_FASE3_FINAL/DeepSORT_Final_Campeao',
}

accs = {name: mm.MOTAccumulator(auto_id=False) for name in TRACKERS_TO_EVALUATE.keys()}
print("Iniciando a Batalha Final: Lendo gabaritos inéditos...")

for seq_name in sequence_folders:
    gt_file_path = os.path.join(GT_DIR, seq_name, 'gt', 'gt.txt')
    if not os.path.exists(gt_file_path): continue

    gt = mm.io.loadtxt(gt_file_path, fmt='mot15-2D').sort_index()

    for tracker_name, tracker_base_dir in TRACKERS_TO_EVALUATE.items():
        ts_file_path = os.path.join(tracker_base_dir, 'txt_files', f'{seq_name}.txt')

        if not os.path.exists(ts_file_path):
            continue

        ts = mm.io.loadtxt(ts_file_path, fmt='mot15-2D').sort_index()

        frame_ids_gt = gt.index.get_level_values('FrameId').unique()
        frame_ids_ts = ts.index.get_level_values('FrameId').unique()
        all_frame_ids = sorted(list(set(frame_ids_gt) | set(frame_ids_ts)))

        for frame_id in all_frame_ids:
            gt_for_frame = gt.loc[frame_id] if frame_id in frame_ids_gt else pd.DataFrame()
            ts_for_frame = ts.loc[frame_id] if frame_id in frame_ids_ts else pd.DataFrame()

            if ts_for_frame.empty:
                dists = np.empty((len(gt_for_frame), 0))
            else:
                dists = mm.distances.iou_matrix(
                    gt_for_frame[['X', 'Y', 'Width', 'Height']],
                    ts_for_frame[['X', 'Y', 'Width', 'Height']],
                    max_iou=0.5
                )

            accs[tracker_name].update(gt_for_frame.index, ts_for_frame.index, dists, frameid=frame_id)

mh = mm.metrics.create()
all_summaries = []

print("\nCalculando Placar Final do TCC...")
for tracker_name, acc in accs.items():
    if not acc.events.empty:
        acc.events.sort_index(inplace=True)
        summary = mh.compute(acc, metrics=mm.metrics.motchallenge_metrics, name=tracker_name)
        all_summaries.append(summary)

if not all_summaries:
    print("\nNenhum resultado encontrado. Os trackers terminaram de rodar?")
else:
    summary_final = pd.concat(all_summaries)

    cols_to_show = ['mota', 'num_false_positives', 'num_misses', 'num_switches']
    summary_filtered = summary_final[[col for col in cols_to_show if col in summary_final.columns]]

    summary_filtered = summary_filtered.rename(columns={
        'mota': 'MOTA (%)',
        'num_false_positives': 'FP',
        'num_misses': 'FN',
        'num_switches': 'IDs'
    })

    if 'MOTA (%)' in summary_filtered.columns:
        summary_filtered['MOTA (%)'] = summary_filtered['MOTA (%)'] * 100

    print("\n\n=================================================================")
    print("🏆 PLACAR FINAL DO TCC - FASE 3: TESTE CEGO (AMBIENTE REAL) 🏆")
    print("=================================================================\n")
    pd.set_option('display.float_format', '{:.1f}'.format)
    print(summary_filtered)

Iniciando a Batalha Final: Lendo gabaritos inéditos...

Calculando Placar Final do TCC...


🏆 PLACAR FINAL DO TCC - FASE 3: TESTE CEGO (AMBIENTE REAL) 🏆

                      MOTA (%)     FP      FN   IDs
1_OcSORT_Campeao          76.7  26598   96304  8702
2_ByteTrack_Campeao       76.8  23525   99685  7811
3_StrongSORT_Campeao      77.1  34211   86965  8196
4_DeepSORT_Campeao        65.3  85867  106051  4209


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ==========================================
# 1. DADOS ATUALIZADOS DA FASE 1
# ==========================================
thresholds = [0.05, 0.3, 0.5]

# Métricas de MOTA (%)
mota_ocsort = [62.0, 80.0, 77.2]
mota_strongsort = [64.9, 80.4, 79.2]
mota_bytetrack = [79.6, 79.4, 76.4]
mota_deepsort = [-43.4, 60.4, 69.7]

# Métricas de Falsos Positivos (FP)
fp_ocsort = [30547, 5769, 2517]
fp_strongsort = [33930, 8282, 3126]
fp_bytetrack = [3304, 2759, 2149]
fp_deepsort = [214192, 43569, 23868]

# Métricas de Falsos Negativos (FN)
fn_ocsort = [23598, 24603, 33444]
fn_strongsort = [17390, 21339, 29480]
fn_bytetrack = [28818, 29816, 35075]
fn_deepsort = [14439, 19293, 24371]

# ==========================================
# 2. CONFIGURAÇÕES DE ESTILO (Cores Vivas e Acadêmicas)
# ==========================================
# Paleta de cores vivas e com alto contraste
color_oc = '#FF8C00'   # Laranja Vibrante
color_st = '#00BFFF'   # Azul Celeste
color_by = '#00C853'   # Verde Esmeralda (Destaca a vitória no 0.05)
color_de = '#E10050'   # Rosa/Magenta (Destaca a queda absurda)

# Estilo global da linha
linha_estilo = {'linewidth': 2, 'marker': 'o', 'markersize': 8}

# ==========================================
# 3. GRÁFICO 1: MOTA vs THRESHOLD
# ==========================================
plt.figure(figsize=(10, 6))

plt.plot(thresholds, mota_ocsort, label='OcSORT', color=color_oc, **linha_estilo)
plt.plot(thresholds, mota_strongsort, label='StrongSORT', color=color_st, **linha_estilo)
plt.plot(thresholds, mota_bytetrack, label='ByteTrack', color=color_by, **linha_estilo)
plt.plot(thresholds, mota_deepsort, label='DeepSORT', color=color_de, **linha_estilo)

# Formatação
plt.title('MOTA por Limiar de Confiança', fontsize=16, pad=15)
plt.xlabel('Threshold (Limiar de Confiança)', fontsize=12)
plt.ylabel('MOTA (%)', fontsize=12)
plt.xticks(thresholds) # Força o eixo X a mostrar só 0.05, 0.3 e 0.5
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(loc='lower right', fontsize=11, framealpha=0.9)

# Salva a imagem em alta resolução
plt.tight_layout()
plt.savefig('mota_por_threshold_atualizado.png', dpi=300, bbox_inches='tight')
plt.show()

# ==========================================
# 4. GRÁFICO 2: FALSOS POSITIVOS vs THRESHOLD
# ==========================================
plt.figure(figsize=(10, 6))

plt.plot(thresholds, fp_ocsort, label='OcSORT', color=color_oc, **linha_estilo)
plt.plot(thresholds, fp_strongsort, label='StrongSORT', color=color_st, **linha_estilo)
plt.plot(thresholds, fp_bytetrack, label='ByteTrack', color=color_by, **linha_estilo)
plt.plot(thresholds, fp_deepsort, label='DeepSORT', color=color_de, **linha_estilo)

# Formatação
plt.title('Falsos Positivos por Limiar de Confiança', fontsize=16, pad=15)
plt.xlabel('Threshold (Limiar de Confiança)', fontsize=12)
plt.ylabel('Quantidade de Falsos Positivos (FP)', fontsize=12)
plt.xticks(thresholds)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(loc='upper right', fontsize=11, framealpha=0.9)

# Formata o eixo Y para não ficar em notação científica (ex: 2e5)
plt.ticklabel_format(style='plain', axis='y')

# Salva a imagem em alta resolução
plt.tight_layout()
plt.savefig('fp_por_threshold_atualizado.png', dpi=300, bbox_inches='tight')
plt.show()

# ==========================================
# 5. GRÁFICO 3: FALSOS NEGATIVOS vs THRESHOLD
# ==========================================
plt.figure(figsize=(10, 6))

plt.plot(thresholds, fn_ocsort, label='OcSORT', color=color_oc, **linha_estilo)
plt.plot(thresholds, fn_strongsort, label='StrongSORT', color=color_st, **linha_estilo)
plt.plot(thresholds, fn_bytetrack, label='ByteTrack', color=color_by, **linha_estilo)
plt.plot(thresholds, fn_deepsort, label='DeepSORT', color=color_de, **linha_estilo)

# Formatação
plt.title('Falsos Negativos por Limiar de Confiança', fontsize=16, pad=15)
plt.xlabel('Threshold (Limiar de Confiança)', fontsize=12)
plt.ylabel('Quantidade de Falsos Negativos (FN)', fontsize=12)
plt.xticks(thresholds)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(loc='lower right', fontsize=11, framealpha=0.9) # Fica no canto inferior para não cobrir as linhas subindo

# Formata o eixo Y para não ficar em notação científica
plt.ticklabel_format(style='plain', axis='y')

# Salva a imagem em alta resolução
plt.tight_layout()
plt.savefig('fn_por_threshold_atualizado.png', dpi=300, bbox_inches='tight')
plt.show()

print("Todos os 3 gráficos gerados e salvos com sucesso!")

In [ ]:
# ==============================================================================
# SCRIPT MESTRE PARA GERAR OS FRAMES DE EXEMPLO (ANÁLISE QUALITATIVA)
# (Roda os 4 melhores trackers em UMA sequência de teste)
# ==============================================================================
import os
import cv2
import torch
import numpy as np
from pathlib import Path
from ultralytics import YOLO
from boxmot.reid.core.reid import ReID


from boxmot.trackers import StrongSort, OcSort, ByteTrack
from deep_sort_realtime.deepsort_tracker import DeepSort

# --- 1. CONFIGURAÇÕES GLOBAIS ---
YOLO_MODEL_PATH = '/content/drive/MyDrive/SoccerNet/Results_augmented/best.pt'
REID_MODEL_PATH = '/content/drive/MyDrive/SoccerNet/bibliotecas_git/osnet_x0_25_msmt17.pt'

BASE_INPUT_DIR  = '/content/drive/MyDrive/SoccerNet/tracking/test/'
BASE_OUTPUT_DIR = '/content/drive/MyDrive/SoccerNet/tracking_visuals_final/'

NOME_DA_SEQUENCIA_DE_TESTE = 'SNMOT-116'

TRACKERS_TO_RUN = [
    #'StrongSort_BEST',
    #'OcSORT_BEST',
    'ByteTrack_BEST',
    #'DeepSort_BEST'
]

# --- FIM DAS CONFIGURAÇÕES ---

def build_tracker(tracker_name, device):

    if tracker_name == 'StrongSort_BEST':
        from boxmot.reid.core.reid import ReID  # caminho correto encontrado
        reid = ReID(
            weights=Path(REID_MODEL_PATH),
            device=device,
            half=False
        )
        return StrongSort(
            reid_model=reid.model,
            max_age=90,
            n_init=5,
            max_cos_dist=0.2
        ), 0.3

    elif tracker_name == 'OcSORT_BEST':
        return OcSort(
            device=device,
            max_age=30,
            asso_threshold=0.2,
            min_hits=3
        ), 0.3

    elif tracker_name == 'ByteTrack_BEST':
        return ByteTrack(
            device=device,
            track_buffer=45,
            track_thresh=0.5,
            match_thresh=0.7
        ), 0.3

    elif tracker_name == 'DeepSort_BEST':
        return DeepSort(
            max_age=30,
            n_init=7,
            max_cosine_distance=0.2
        ), 0.5

    else:
        raise ValueError(f"Tracker desconhecido: {tracker_name}")


def generate_all_sample_frames():

    assert Path(YOLO_MODEL_PATH).exists(), f"YOLO não encontrado: {YOLO_MODEL_PATH}"
    assert Path(REID_MODEL_PATH).exists(), f"ReID não encontrado: {REID_MODEL_PATH}"
    print("✅ Caminhos de modelo verificados.")

    DEVICE = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')
    print(f"Usando dispositivo: {DEVICE}")

    print("Carregando modelo YOLOv8...")
    yolo_model = YOLO(YOLO_MODEL_PATH)
    yolo_model.to(DEVICE)
    print("Modelo YOLO carregado.")

    # --- Cores por ID ---
    track_colors = {}
    np.random.seed(42)

    def get_color_for_id(track_id):
        if track_id not in track_colors:
            track_colors[track_id] = tuple(np.random.randint(50, 255, 3).tolist())
        return track_colors[track_id]

    # --- LOOP 1: Por Tracker ---
    for tracker_name in TRACKERS_TO_RUN:
        print(f"\n=======================================================")
        print(f"--- PROCESSANDO COM: {tracker_name} ---")
        print(f"=======================================================")

        tracker_instance, conf_thresh = build_tracker(tracker_name, DEVICE)

        output_frame_dir = os.path.join(BASE_OUTPUT_DIR, tracker_name, NOME_DA_SEQUENCIA_DE_TESTE,"03")
        os.makedirs(output_frame_dir, exist_ok=True)
        print(f"Salvando frames em: {output_frame_dir}")
        print(f"  Processando Sequência: {NOME_DA_SEQUENCIA_DE_TESTE}...")

        current_image_dir = os.path.join(BASE_INPUT_DIR, NOME_DA_SEQUENCIA_DE_TESTE, 'img1')
        image_files = sorted([f for f in os.listdir(current_image_dir) if f.lower().endswith('.jpg')])

        # --- LOOP 2: Por Frame ---
        for frame_num, image_name in enumerate(image_files):
            if (frame_num + 1) % 200 == 0:
                print(f"    ...frame {frame_num + 1}/{len(image_files)}")

            frame_orig = cv2.imread(os.path.join(current_image_dir, image_name))
            if frame_orig is None:
                continue

            # 1. DETECÇÃO
            yolo_results = yolo_model.predict(frame_orig, device=DEVICE, verbose=False, conf=conf_thresh)
            frame_to_draw = frame_orig.copy()

            # 2. RASTREAMENTO

            if tracker_name == 'DeepSort_BEST':
                detections_for_tracker = []
                for box in yolo_results[0].boxes:
                    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                    conf       = box.conf[0].cpu().item()
                    class_name = yolo_model.names[int(box.cls[0].cpu().item())]
                    bbox_xywh  = [x1, y1, (x2 - x1), (y2 - y1)]  # corrigido: largura = x2-x1
                    detections_for_tracker.append((bbox_xywh, conf, class_name))

                tracks = tracker_instance.update_tracks(detections_for_tracker, frame=frame_orig)

                for track in tracks:
                    if not track.is_confirmed():
                        continue
                    track_id = track.track_id
                    x1, y1, x2, y2 = map(int, track.to_tlbr())
                    cls_name = track.get_det_class()
                    color    = get_color_for_id(int(track_id))
                    label    = f'ID:{int(track_id)} {cls_name}'
                    cv2.rectangle(frame_to_draw, (x1, y1), (x2, y2), color, 2)
                    cv2.putText(frame_to_draw, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

            else:  # StrongSort, OcSort, ByteTrack (BoxMOT)
                detections = yolo_results[0].boxes.data.cpu().numpy()
                if detections.shape[0] > 0:
                    tracks = tracker_instance.update(detections, frame_orig)
                else:
                    tracks = tracker_instance.update(np.empty((0, 6)), frame_orig)

                if len(tracks) > 0:
                    for track in tracks:
                        x1, y1, x2, y2, track_id, conf_t, cls, idx = track
                        color = get_color_for_id(int(track_id))
                        label = f'ID:{int(track_id)} {yolo_model.names[int(cls)]}'
                        cv2.rectangle(frame_to_draw, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
                        cv2.putText(frame_to_draw, label, (int(x1), int(y1) - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

            # 3. SALVA FRAME
            cv2.imwrite(os.path.join(output_frame_dir, image_name), frame_to_draw)

        print(f"  -> Frames para {NOME_DA_SEQUENCIA_DE_TESTE} salvos.")

    print("\n--- PROCESSAMENTO GERAL CONCLUÍDO! ---")


if __name__ == '__main__':
    generate_all_sample_frames()

In [ ]:
# ==============================================================================
# SCRIPT PARA GERAR OS 24 VÍDEOS (6 sequências x 4 trackers)
# ==============================================================================
import os
import cv2
from pathlib import Path

# --- CONFIGURAÇÕES ---
BASE_FRAMES_DIR = '/content/drive/MyDrive/SoccerNet/tracking_visuals_final/'
BASE_OUTPUT_DIR = '/content/drive/MyDrive/SoccerNet/tracking_videos_final/'

TRACKERS = [
    'StrongSort_BEST',
    'OcSORT_BEST',
    'ByteTrack_BEST',
    'DeepSort_BEST'
]

SEQUENCIAS = [
    'SNMOT-116',
    'SNMOT-117',
    'SNMOT-118',
    'SNMOT-119',
    'SNMOT-121',
    'SNMOT-122'
]

FPS = 25  # FPS padrão do SoccerNet
# --- FIM DAS CONFIGURAÇÕES ---

def gerar_videos():
    os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)
    total = len(TRACKERS) * len(SEQUENCIAS)
    count = 0

    for tracker in TRACKERS:
        for sequencia in SEQUENCIAS:
            count += 1
            frames_dir = os.path.join(BASE_FRAMES_DIR, tracker, sequencia)

            if not os.path.exists(frames_dir):
                print(f"[{count}/{total}] ⚠️  Pasta não encontrada, pulando: {frames_dir}")
                continue

            image_files = sorted([
                f for f in os.listdir(frames_dir)
                if f.lower().endswith('.jpg')
            ])

            if not image_files:
                print(f"[{count}/{total}] ⚠️  Nenhum frame encontrado em: {frames_dir}")
                continue

            # Lê o primeiro frame para pegar as dimensões
            first_frame = cv2.imread(os.path.join(frames_dir, image_files[0]))
            h, w = first_frame.shape[:2]

            # Nome do vídeo de saída
            video_name = f"{tracker}_{sequencia}.mp4"
            video_path = os.path.join(BASE_OUTPUT_DIR, video_name)

            writer = cv2.VideoWriter(
                video_path,
                cv2.VideoWriter_fourcc(*'mp4v'),
                FPS,
                (w, h)
            )

            for image_name in image_files:
                frame = cv2.imread(os.path.join(frames_dir, image_name))
                if frame is not None:
                    writer.write(frame)

            writer.release()
            print(f"[{count}/{total}] ✅ {video_name}")

    print(f"\n--- {total} VÍDEOS GERADOS EM: {BASE_OUTPUT_DIR} ---")

if __name__ == '__main__':
    gerar_videos()